# Disease detection, symptom diagnosis, and severity analysis

Load saved classifiers, calculate symptom attributes, and inspect severity aggregation and evaluation variants. The disease-detector training code is not included; its pretrained weights are supplied in the separate model archive.

**Research code with separate pretrained artifacts.** Extract the model Release asset at the repository root. No images, labels, feature tables, or saved outputs are included. Read [the execution guide](../docs/EXECUTION.md) and [review notes](../docs/REVIEW_NOTES.md) before running experiments.

Paper: https://doi.org/10.1007/s44196-026-01458-x

## Path configuration
Set the project root below. Dataset requests: naoum1496@gmai.com.

In [ ]:
from pathlib import Path
import os

# Run from the repository root or its notebooks/ directory.
# For another location, set STRAWBERRY_PROJECT_ROOT or edit PROJECT_ROOT.
working_dir = Path.cwd()
default_root = working_dir.parent if working_dir.name == "notebooks" else working_dir
PROJECT_ROOT = Path(os.environ.get("STRAWBERRY_PROJECT_ROOT", str(default_root))).expanduser().resolve()
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FEATURE_DIR = DATA_DIR / "features"
for directory in (MODEL_DIR, MODEL_DIR / "SVM_PCA_GLCM", RESULTS_DIR, FEATURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)


## 1. Imports and dataset inventory

In [ ]:
import numpy as np
import cv2
from keras.preprocessing import image
from keras.applications.inception_v3 import preprocess_input
import matplotlib.pyplot as plt
import random
import os

In [ ]:
import os
import glob
import matplotlib.pyplot as plt

# Modify this path as needed
path = f"{DATA_DIR}/All_Included_Final_Severity_Stage"

# Get subfolders
subfolders = [name for name in os.listdir(path) if os.path.isdir(os.path.join(path, name))]

# Iterate through each subfolder
for subfolder in subfolders:
    subfolder_path = os.path.join(path, subfolder)

    # Get image files, including case-insensitive extensions
    image_extensions = [".jpg", ".JPG", ".png", ".PNG"]
    image_files = []
    for extension in image_extensions:
        image_files += glob.glob(os.path.join(subfolder_path, f"*{extension}"))

    # Count images
    num_images = len(image_files)
    print(f"Subfolder: {subfolder}, Number of images: {num_images}")
    # Display 3 images (max 3 images per folder)
    for i in range(min(3, num_images)):
        # Read image
        img = plt.imread(image_files[i])
        # Create subplot and display image
        plt.subplot(1, 3, i + 1)
        plt.imshow(img)
        plt.axis("off")

    # Show images for this subfolder
    plt.suptitle(f"Subfolder: {subfolder}")
    plt.show()
    plt.close()  # Clear plot for next subfolder



## 2. Load required model artifacts

In [ ]:
import tensorflow as tf
from joblib import load

# Replace 'path_to_your_saved_model' with the actual path to your saved model directory.
model = tf.keras.models.load_model(f'{MODEL_DIR}/Clean_Reviwers_DisDetect_best_final_model.h5')
# Load the pre-trained model
model_2 = tf.keras.models.load_model(f'{MODEL_DIR}/Spots_Distribution_best_model_overall.h5')
scaler = load(f'{MODEL_DIR}/SVM_PCA_GLCM/NG_scaler.joblib')
model_3 = load(f'{MODEL_DIR}/SVM_PCA_GLCM/NG_random_forest_model.joblib')
# Load the scaler and model from their respective saved .joblib files
#scaler = load(f'{MODEL_DIR}/SVM_PCA_GLCM/NG_scaler.joblib')


## 3. Colour and spot features

In [ ]:
def load_and_resize_image(img, new_size=(224, 224)):
    # Resize the image
    resized_img = cv2.resize(img, new_size)
    return resized_img

# Function to extract the foreground by creating a mask to exclude the black background
def extract_foreground(image):
    grayscale = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    foreground_mask = grayscale > 0
    foreground = cv2.bitwise_and(image, image, mask=foreground_mask.astype(np.uint8))
    return foreground, foreground_mask
def preprocess_shiny(image):
    # Apply Gaussian Blur
    blurred = cv2.GaussianBlur(image, (5, 5), 0)

    # Convert to HSV and split the channels
    hsv = cv2.cvtColor(blurred, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)

    # Apply CLAHE to the Value channel
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    v = clahe.apply(v)

    # Merge the HSV channels back together
    hsv_clahe = cv2.merge([h, s, v])

    # Convert back to BGR
    result = cv2.cvtColor(hsv_clahe, cv2.COLOR_HSV2BGR)
    return result
# Function to calculate the peak of the green color in the image
def calculate_green_peak(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hist_hue = cv2.calcHist([hsv], [0], None, [180], [0, 180])
    green_peak = np.argmax(hist_hue[25:85]) + 25
    return green_peak

# Function to determine the lower bound of the green color based on the peak
def set_lower_green(green_peak):
    if green_peak <= 29:
        a = 25
    elif 30 <= green_peak <= 34:
        a = green_peak
    elif 35 <= green_peak <= 39:
        a = 30
    elif 40 <= green_peak <= 43:
        a = 35
    elif 44 <= green_peak <= 50:
        a = 25
    elif green_peak >= 51:
        a = 30
    else:
        a = green_peak - 10
    return a

# Function to threshold the image for green regions
def threshold_green(image, green_peak):
    a = set_lower_green(green_peak)
    lower_green = np.array([a, 30, 30])
    upper_green = np.array([85, 255, 255])
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    green_mask = cv2.inRange(hsv, lower_green, upper_green)
    return green_mask

# Function to separate and refine green and non-green regions
def separate_regions(image, green_mask):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    green_mask = cv2.morphologyEx(green_mask, cv2.MORPH_OPEN, kernel)
    green_mask = cv2.morphologyEx(green_mask, cv2.MORPH_CLOSE, kernel)
    non_green_mask = cv2.bitwise_not(green_mask)
    green_region = cv2.bitwise_and(image, image, mask=green_mask)
    non_green_region = cv2.bitwise_and(image, image, mask=non_green_mask)
    return green_region, non_green_region


# Function to calculate area percentages
def calculate_area_percentages(green_mask, non_green_region, foreground_mask):
    green_area = np.sum(green_mask > 0)
    # Calculate the non-green area by subtracting the green mask from the foreground mask
    non_green_area = np.sum((foreground_mask > 0) & (green_mask == 0))

    total_area = np.sum(foreground_mask > 0)
    percentage_green = (green_area / total_area) * 100
    percentage_non_green = (non_green_area / total_area) * 100
    return percentage_green, percentage_non_green, green_area, non_green_area

def classify_plant_health(non_green_region):

    # Case 1: Early
    if (non_green_region <= 25): #before:20
        return "Slightly-infected"

    # Case 2: Medium Stage
    elif ((25 < non_green_region <= 55)): #before ago 45,,,,,,-> 50
        return "Moderately-infected"

    # Case 3: Medium Stage
    elif (55< non_green_region):
        return "Highly-infected"

    # If none of the cases match, return undefined
    else:
        return "Undefined"
def classify_plant_stress(green_region):

    # Case 1: Severe
    if (green_region <=40 ): #before:50,55
        return "High"

    # Case 2: Medium Stage
    elif ((40 < green_region <= 75)):  # before:80
        return "Medium"

    # Case 3: Early Stage
    elif (75 < green_region):
        return "Low"

    # If none of the cases match, return undefined
    else:
        return "Undefined"
# Function to segment spots based on color ranges
def segment_spots(hsv_image):

    # Define ranges for different colors in the HSV space
    brown_lower = np.array([10, 100, 20])
    brown_upper = np.array([20, 255, 200])
    red_lower = np.array([0, 100, 20])
    red_upper = np.array([25, 255, 255]) #updated#......changed in red1
    #red_upper = np.array([10, 255, 255])
    reddish_brown_lower = np.array([0, 100, 50])
    reddish_brown_upper = np.array([20, 255, 150])


    # Create masks for each color range and combine them
    brown_mask = cv2.inRange(hsv_image, brown_lower, brown_upper)
    red_mask = cv2.inRange(hsv_image, red_lower, red_upper)
    reddish_brown_mask = cv2.inRange(hsv_image, reddish_brown_lower, reddish_brown_upper)

    combined_mask = brown_mask | red_mask | reddish_brown_mask

    return combined_mask
def calculate_spot_density(combined_mask, hsv_image):
    # Create a mask for the leaf area (foreground)
    leaf_area_mask = cv2.inRange(hsv_image, np.array([0, 0, 1]), np.array([179, 255, 255]))
    leaf_area_pixels = cv2.countNonZero(leaf_area_mask)
    spot_pixels = cv2.countNonZero(combined_mask)
    density = spot_pixels / leaf_area_pixels if leaf_area_pixels > 0 else 0
    return density


## 4. GLCM texture features

In [ ]:
from skimage.feature import graycomatrix, graycoprops
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
def calculate_entropy(glcm):
    """
    Calculate entropy from the GLCM.
    """
    with np.errstate(divide='ignore', invalid='ignore'):
        entropy = -np.sum(glcm * np.log2(glcm + (glcm == 0)), axis=(0, 1))
    return entropy


# You will need to define or import your calculate_entropy function here as well.
# from your_module import calculate_entropy

def analyze_texture(image, distances=[1], angles=[0]):
    """
    Analyzes the texture of an image using GLCM without assigning labels.
    Returns a dictionary of GLCM features including contrast, energy, homogeneity, correlation, dissimilarity, and entropy.
    """
    # Normalize image to [0, 255] and convert to uint8
    image = (image.astype("float32") / image.max() * 255).astype('uint8')

    # Compute GLCM
    glcm = graycomatrix(image, distances=distances, angles=angles, levels=256, symmetric=True, normed=True)

    # Extract GLCM features
    features = {}
    for prop in ['contrast', 'homogeneity', 'energy', 'correlation', 'dissimilarity']:
        f = graycoprops(glcm, prop)
        for a, angle in enumerate(angles):
            for d, distance in enumerate(distances):
                features[f"{prop}_d{distance}_a{angle}"] = f[d, a]
    # Calculate entropy
    entropy = calculate_entropy(glcm)
    for a, angle in enumerate(angles):
        for d, distance in enumerate(distances):
            features[f"entropy_d{distance}_a{angle}"] = entropy[d, a]

    return features


## 5. Original majority-voting helper

In [ ]:

def majority_voting(health_pred, stress_pred, density_pred, spots_pred, texture_pred):
    predictions = [health_pred, stress_pred, density_pred, spots_pred, texture_pred]
    prediction_count = {pred: predictions.count(pred) for pred in set(predictions)}

    # If all predictions are the same, return that prediction.
    if len(set(predictions)) == 1:
        return health_pred

    # Sort predictions by count in descending order to find the majority.
    sorted_predictions = sorted(prediction_count.items(), key=lambda x: x[1], reverse=True)

    # If there's a clear majority, return that prediction.
    if sorted_predictions[0][1] > 1:
        return sorted_predictions[0][0]

    # If there's no clear majority, return texture_pred.
    return texture_pred


## 6. Evaluation plotting helpers

In [ ]:
import seaborn as sns  # For confusion matrix visualization
from sklearn.metrics import confusion_matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# After processing all images and before printing accuracy scores
def plot_confusion_matrix(actual_labels, final_predictions, class_labels):
    """
    Generates and plots a confusion matrix for the given actual and predicted labels.

    Parameters:
    - actual_labels: list of actual labels
    - final_predictions: list of predicted labels by the model
    - class_labels: list of unique class labels
    """
    # Generate the confusion matrix
    matrix = confusion_matrix(actual_labels, final_predictions, labels=class_labels)

    # Plotting using seaborn for better visualization
    plt.figure(figsize=(2, 2))
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)
    #plt.title('Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('Actual Label')
    plt.show()

def plot_Bconfusion_matrix(y_true, y_pred, classes, title='Confusion Matrix'):
    plt.figure(figsize=(1, 1))
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(cmap=plt.cm.Blues)
    plt.title(title)
    plt.show()

## 7. Majority-first voting variant

This helper chooses a majority before considering weights. Its dictionary also collapses repeated labels. It is preserved without scientific changes; see the review notes.

In [ ]:
def majority_voting_1(health_pred, stress_pred, density_pred, spots_pred, texture_pred):
    # Define the weights based on model accuracy.
    weights = {
        'health_pred': 94.13645,
        'stress_pred': 94.81301,
        'density_pred': 94.43714,
        'spots_pred': 84.11953,
        'texture_pred': 96.84270
    }

    predictions = [health_pred, stress_pred, density_pred, spots_pred, texture_pred]
    prediction_count = {pred: predictions.count(pred) for pred in set(predictions)}

    # If all predictions are the same, return that prediction.
    if len(set(predictions)) == 1:
        return health_pred

    # Sort predictions by count in descending order to find the majority.
    sorted_predictions = sorted(prediction_count.items(), key=lambda x: x[1], reverse=True)

    # If there's a clear majority, return that prediction.
    if sorted_predictions[0][1] > 1:
        return sorted_predictions[0][0]

    # If there's no clear majority, use weighted voting to decide.
    weighted_predictions = {
        health_pred: weights['health_pred'],
        stress_pred: weights['stress_pred'],
        density_pred: weights['density_pred'],
        spots_pred: weights['spots_pred'],
        texture_pred: weights['texture_pred']
    }

    prediction_scores = {}
    for pred, weight in weighted_predictions.items():
        if pred in prediction_scores:
            prediction_scores[pred] += weight
        else:
            prediction_scores[pred] = weight

    # Sort by weighted score and return the prediction with the highest score.
    sorted_weighted_predictions = sorted(prediction_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_weighted_predictions[0][0]


## 8. Evaluate the majority-first variant

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

# Assumed predefined functions:
# extract_foreground, preprocess_shiny, calculate_green_peak, threshold_green, separate_regions,
# calculate_area_percentages, classify_plant_health, classify_plant_stress, segment_spots, calculate_spot_density,
# load_and_resize_image, preprocess_input, model

def process_images_in_folder(root_folder_path):
    subfolders = [os.path.join(root_folder_path, f) for f in os.listdir(root_folder_path) if os.path.isdir(os.path.join(root_folder_path, f))]
    health_labels = []
    health_predictions = []
    stress_labels = []
    stress_predictions = []
    density_labels = []
    density_predictions = []
    spots_labels = []
    spots_predictions = []
    texture_labels = []
    texture_predictions = []
    final_predictions = []  # Store final predictions after majority voting
    actual_labels = []  # Store actual labels for accuracy calculation
    count = 0
    # Variables for binary classification (Healthy vs. Unhealthy)
    binary_predictions = []
    binary_actual_labels = []
    i=0


    for subfolder in subfolders:
        actual_stage = os.path.basename(subfolder)  # The actual stage is the subfolder name
        if actual_stage == 'Healthy':
           actual_binary_label = 'Healthy'
        else:
          actual_binary_label = 'Unhealthy'

        files = [f for f in os.listdir(subfolder) if os.path.isfile(os.path.join(subfolder, f))]
        image_files = [f for f in files if f.lower().endswith(('.png', '.jpg','.JPG', '.jpeg'))]

        for image_file in image_files:
            image_path = os.path.join(subfolder, image_file)
            count += 1
            i+=1
            print(i)

            im = cv2.imread(image_path)
            image = load_and_resize_image(im)
            image_2 = image.copy()
            image = np.expand_dims(image, axis=0)
            image = preprocess_input(image)
            prediction = model.predict(image)
            threshold = 0.5
            predicted_class = 'Healthy' if prediction[0][0] < threshold else 'Unhealthy'
                    # Record binary predictions and actual labels

            binary_actual_labels.append(actual_binary_label)
            binary_predictions.append(predicted_class)


            if predicted_class == 'Healthy':
                severity_mapping_1 = 'Healthy'
                severity_stage_G = 'Healthy'
                severity_stage_2 = 'Healthy'
                severity_levels_spots='Healthy'
                severity_texture='Healthy'
                # Assuming that 'NULL' maps to 'Healthy'
                health_predictions.append(severity_mapping_1)  # Or 'Healthy' if you have such a label
                stress_predictions.append(severity_stage_G )  # Same assumption
                density_predictions.append(severity_stage_2)  # Same assumption
                spots_predictions.append(severity_levels_spots)  # Same assumption
                texture_predictions.append(severity_texture)  # Same assumption



            else:
                foreground, foreground_mask = extract_foreground(image_2)
                preprocessed_foreground = preprocess_shiny(foreground)

                # Calculate the green color peak on the foreground
                green_peak = calculate_green_peak(preprocessed_foreground)
                # Set the green threshold and obtain the green mask for the foreground
                green_mask = threshold_green(foreground, green_peak)

                # Separate and refine the green and non-green regions from the foreground
                green_region, non_green_region = separate_regions(foreground, green_mask)

                # Calculate area percentages
                percentage_green, percentage_non_green, green_area, non_green_area = calculate_area_percentages(green_mask, non_green_region, foreground_mask)

                if non_green_region.ndim == 3:
                  image3 = cv2.cvtColor(non_green_region, cv2.COLOR_BGR2GRAY)

                # Analyze texture
                results = analyze_texture(image3)
                # Define the mapping from numerical prediction to categorical label
                stage_mapping = {0:'Early_Stage', 1: 'Medium_Stage', 2: 'Severe_Stage'}
                # Convert the dictionary of features to an array-like structure
                features_array = np.array(list(results.values())).reshape(1, -1)
                # Normalize the features using the loaded scaler
                scaled_features = scaler.transform(features_array)
                #print(scaled_features.shape)  # Check the current shape
                # Assuming your model expects input of shape (1, height, width, channels)
                # You may need to reshape or adjust your input here.

                # Use the model to predict using the normalized features
                # Use the model to predict using the normalized features
                numeric_prediction = model_3.predict(scaled_features)

                # Map the numerical prediction to a categorical label
                texture_prediction = stage_mapping[numeric_prediction[0]]
                #print(texture_prediction)
                severity_texture=texture_prediction




                 #Infection Label classification

                Label_1 = classify_plant_health(percentage_non_green)
                severity_mapping_1 = {
                    'Slightly-infected': 'Early_Stage',
                    'Moderately-infected': 'Medium_Stage',
                    'Highly-infected': 'Severe_Stage',
                }
                severity_stage_1 = severity_mapping_1.get(Label_1, 'Undefined Stage')

                #Stress Label classification
                Label_G = classify_plant_stress(percentage_green)
                severity_mapping_G = {
                    'Low': 'Early_Stage',
                    'Medium': 'Medium_Stage',
                    'High': 'Severe_Stage',
                }
                severity_stage_G = severity_mapping_G.get(Label_G, 'Undefined Stage')

                hsv_image = cv2.cvtColor(image_2, cv2.COLOR_BGR2HSV)
                combined_mask = segment_spots(hsv_image)
                density = calculate_spot_density(combined_mask, hsv_image)

                # Define thresholds for density labeling
                low_threshold = 0.022  # Placeholder value for low density->0.023
                moderate_threshold = 0.17  # Placeholder value for moderate density ->0.17
                Label_2 = 'Low' if density < low_threshold else ('Moderate' if density < moderate_threshold else 'High')

                severity_mapping = {
                    'Low': 'Early_Stage',
                    'Moderate': 'Medium_Stage',
                    'High': 'Severe_Stage',
                }

                severity_stage_2 = severity_mapping[Label_2]

                # Define categories
                #Check Spots Distribution
                categories = ['Clustered', 'Scattered', 'Speckled', 'Varied']

                # Define severity levels
                severity_levels_spots = {
                    'Scattered': 'Early_Stage',
                    'Speckled': 'Medium_Stage',
                    'Clustered': 'Severe_Stage',
                    'Varied': 'Severe_Stage'
                }
                # Make prediction
                prediction = model_2.predict(image)

                # Convert probabilities to class label
                predicted_class_index = np.argmax(prediction)
                predicted_category = categories[predicted_class_index]

                # Map severity
                severity_Spots = severity_levels_spots.get(predicted_category, 'Unknown')

            # Check if severity stages are the same and increment the counter
                health_predictions.append(severity_stage_1)
                stress_predictions.append(severity_stage_G)
                density_predictions.append(severity_stage_2)
                spots_predictions.append(severity_Spots)
                texture_predictions.append(texture_prediction)

            # Add actual labels for accuracy calculation
            health_labels.append(actual_stage)
            stress_labels.append(actual_stage)
            density_labels.append(actual_stage)
            spots_labels.append(actual_stage)
            texture_labels.append(actual_stage)
            actual_labels.append(actual_stage)


            # Majority voting to determine final prediction
            final_prediction = majority_voting_1(severity_stage_1, severity_stage_G, severity_stage_2,severity_Spots,severity_texture)
            final_predictions.append(final_prediction)



            if count >= 6000:  # Stopping condition
                break

        if count >= 6000:
            break

    # Binary classification labels
    binary_class_labels = ['Healthy', 'Unhealthy']
    # Plotting confusion matrix for binary classification
    plot_Bconfusion_matrix(binary_actual_labels, binary_predictions, binary_class_labels, title='Healthy vs. Unhealthy Classification')
    Binary_accuracy = accuracy_score(binary_actual_labels , binary_predictions)
    # Calculate precision, recall, and F1-score for binary classification
    binary_precision, binary_recall, binary_f1_score, _ = precision_recall_fscore_support(binary_actual_labels, binary_predictions, average='binary', pos_label='Unhealthy')
    print('\n')
    print(classification_report(binary_actual_labels, binary_predictions, target_names=binary_class_labels))

    # Print precision, recall, and F1-score for binary classification
    print(f'Binary Classification Precision (Unhealthy): {binary_precision}')
    print(f'Binary Classification Recall (Unhealthy): {binary_recall}')
    print(f'Binary Classification F1-Score (Unhealthy): {binary_f1_score}')


    # Accuracy calculations
    health_accuracy = accuracy_score(health_labels, health_predictions)
    print('\n\n Infection Level Confusion Matrix')
    class_labels = ['Healthy','Early_Stage', 'Medium_Stage', 'Severe_Stage']  # Adjust labels as necessary
    plot_confusion_matrix(health_labels, health_predictions, class_labels)
    print('\n')
    print(classification_report(health_labels, health_predictions, target_names=class_labels))
    # Calculate precision, recall, and F1-score for Infection Level Classification
    health_precision, health_recall, health_f1_score, _ = precision_recall_fscore_support(health_labels, health_predictions, average='weighted')

    # Print out precision, recall, and F1-score for Infection Level Classification
    print(f'Infection Level Classification Precision: {health_precision}')
    print(f'Infection Level Classification Recall: {health_recall}')
    print(f'Infection Level Classification F1-Score: {health_f1_score}')

    # Then follow with your accuracy calculations and printing
    print('\n\n Leaf Stress Confusion Matrix')
    stress_accuracy = accuracy_score(stress_labels, stress_predictions)
    plot_confusion_matrix(stress_labels, stress_predictions, class_labels)
    print('\n')
    print(classification_report(stress_labels, stress_predictions, target_names=class_labels))
    # Calculate precision, recall, and F1-score for Leaf Stress Classification
    stress_precision, stress_recall, stress_f1_score, _ = precision_recall_fscore_support(stress_labels, stress_predictions, average='weighted')

    # Print precision, recall, and F1-score for Leaf Stress Classification
    print(f'Leaf Stress Classification Precision: {stress_precision}')
    print(f'Leaf Stress Classification Recall: {stress_recall}')
    print(f'Leaf Stress Classification F1-Score: {stress_f1_score}')

    print('\n\n Spots Density Confusion Matrix')
    density_accuracy = accuracy_score(density_labels, density_predictions)
    plot_confusion_matrix(density_labels, density_predictions, class_labels)
    print('\n')
    print(classification_report(density_labels, density_predictions, target_names=class_labels))

    # Calculate precision, recall, and F1-score for Spots Density Classification
    density_precision, density_recall, density_f1_score, _ = precision_recall_fscore_support(density_labels, density_predictions, average='weighted')

    # Print out precision, recall, and F1-score for Spots Density Classification
    print(f'Spots Density Classification Precision: {density_precision}')
    print(f'Spots Density Classification Recall: {density_recall}')
    print(f'Spots Density Classification F1-Score: {density_f1_score}')

    print('\n\n Texture Analysis Confusion Matrix')
    texture_accuracy = accuracy_score(texture_labels, texture_predictions)
    plot_confusion_matrix(texture_labels, texture_predictions, class_labels)
    print('\n')
    print(classification_report(texture_labels, texture_predictions, target_names=class_labels))
    # Calculate precision, recall, and F1-score for Texture Analysis Classification
    texture_precision, texture_recall, texture_f1_score, _ = precision_recall_fscore_support(texture_labels, texture_predictions, average='weighted')

    # Print precision, recall, and F1-score for Texture Analysis Classification
    print(f'Texture Analysis Classification Precision: {texture_precision}')
    print(f'Texture Analysis Classification Recall: {texture_recall}')
    print(f'Texture Analysis Classification F1-Score: {texture_f1_score}')

    print('\n\n Spots Distribution Confusion Matrix')
    spots_accuracy = accuracy_score(spots_labels, spots_predictions)
    plot_confusion_matrix(spots_labels, spots_predictions, class_labels)
    print('\n')
    print(classification_report(spots_labels, spots_predictions, target_names=class_labels))
    # Calculate precision, recall, and F1-score for Spots Distribution Classification
    spots_precision, spots_recall, spots_f1_score, _ = precision_recall_fscore_support(spots_labels, spots_predictions, average='weighted')

    # Print out precision, recall, and F1-score for Spots Distribution Classification
    print(f'Spots Distribution Classification Precision: {spots_precision}')
    print(f'Spots Distribution Classification Recall: {spots_recall}')
    print(f'Spots Distribution Classification F1-Score: {spots_f1_score}')

    print('\n\n Majority Voting Confusion Matrix')
    # Calculate and print accuracy
    final_accuracy = accuracy_score(actual_labels, final_predictions)
    plot_confusion_matrix(actual_labels, final_predictions, class_labels)
    print('\n')
    print(classification_report(actual_labels, final_predictions, target_names=class_labels))
    # Calculate precision, recall, and F1-score for Majority Voting Classification
    final_precision, final_recall, final_f1_score, _ = precision_recall_fscore_support(actual_labels, final_predictions, average='weighted')

    # Print out precision, recall, and F1-score for Majority Voting Classification
    print(f'Majority Voting Classification Precision: {final_precision}')
    print(f'Majority Voting Classification Recall: {final_recall}')
    print(f'Majority Voting Classification F1-Score: {final_f1_score}')
    print('\n\n')

    # Visualization call (add this)

    print(f'Binary Classification Healthy_Unhealthy Accuracy: {Binary_accuracy}')
    print(f'Infection Level Accuracy: {health_accuracy}')
    print(f'Stress Classification Accuracy: {stress_accuracy}')
    print(f'Density Classification Accuracy: {density_accuracy}')
    print(f'Spots Distribution Accuracy: {spots_accuracy}')
    print(f'Texture Analysis Accuracy: {texture_accuracy}')
    print(f'Final Classification Accuracy: {final_accuracy}')
    print(f'Total Image: {count}')

    return {
        'total_images_processed': count,
        'health_accuracy': health_accuracy,
        'stress_accuracy': stress_accuracy,
        'density_accuracy': density_accuracy,
        'spots_accuracy': spots_accuracy,
        'texture_accuracy': texture_accuracy,
        'final_accuracy': final_accuracy
    }
# Specify the path to your image folder here
root_folder_path = f'{DATA_DIR}/All_Included_Final_Severity_Stage'
results = process_images_in_folder(root_folder_path)


## 9. Collect predictions for fold summaries

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

# Assumed predefined functions (you must have them defined earlier in the notebook):
# extract_foreground, preprocess_shiny, calculate_green_peak, threshold_green, separate_regions,
# calculate_area_percentages, classify_plant_health, classify_plant_stress, segment_spots, calculate_spot_density,
# load_and_resize_image, preprocess_input, model, model_2, model_3, scaler, analyze_texture, majority_voting_1,
# plot_Bconfusion_matrix, plot_confusion_matrix

def process_images_in_folder(root_folder_path):
    subfolders = [os.path.join(root_folder_path, f) for f in os.listdir(root_folder_path) if os.path.isdir(os.path.join(root_folder_path, f))]
    health_labels = []
    health_predictions = []
    stress_labels = []
    stress_predictions = []
    density_labels = []
    density_predictions = []
    spots_labels = []
    spots_predictions = []
    texture_labels = []
    texture_predictions = []
    final_predictions = []
    actual_labels = []
    count = 0
    binary_predictions = []
    binary_actual_labels = []
    i = 0

    for subfolder in subfolders:
        actual_stage = os.path.basename(subfolder)
        actual_binary_label = 'Healthy' if actual_stage == 'Healthy' else 'Unhealthy'
        files = [f for f in os.listdir(subfolder) if os.path.isfile(os.path.join(subfolder, f))]
        image_files = [f for f in files if f.lower().endswith(('.png', '.jpg','.JPG', '.jpeg'))]

        for image_file in image_files:
            image_path = os.path.join(subfolder, image_file)
            count += 1
            i += 1
            print(i)

            im = cv2.imread(image_path)
            image = load_and_resize_image(im)
            image_2 = image.copy()
            image = np.expand_dims(image, axis=0)
            image = preprocess_input(image)
            prediction = model.predict(image, verbose=0)
            threshold = 0.5
            predicted_class = 'Healthy' if prediction[0][0] < threshold else 'Unhealthy'

            binary_actual_labels.append(actual_binary_label)
            binary_predictions.append(predicted_class)

            if predicted_class == 'Healthy':
                health_predictions.append('Healthy')
                stress_predictions.append('Healthy')
                density_predictions.append('Healthy')
                spots_predictions.append('Healthy')
                texture_predictions.append('Healthy')
            else:
                foreground, foreground_mask = extract_foreground(image_2)
                preprocessed_foreground = preprocess_shiny(foreground)

                green_peak = calculate_green_peak(preprocessed_foreground)
                green_mask = threshold_green(foreground, green_peak)
                green_region, non_green_region = separate_regions(foreground, green_mask)

                percentage_green, percentage_non_green, green_area, non_green_area = calculate_area_percentages(green_mask, non_green_region, foreground_mask)

                if non_green_region.ndim == 3:
                    image3 = cv2.cvtColor(non_green_region, cv2.COLOR_BGR2GRAY)

                # Analyze texture
                results = analyze_texture(image3)
                stage_mapping = {0:'Early_Stage', 1: 'Medium_Stage', 2: 'Severe_Stage'}
                features_array = np.array(list(results.values())).reshape(1, -1)
                scaled_features = scaler.transform(features_array)
                numeric_prediction = model_3.predict(scaled_features)
                texture_prediction = stage_mapping[numeric_prediction[0]]
                severity_texture = texture_prediction

                # Infection Label classification
                Label_1 = classify_plant_health(percentage_non_green)
                severity_mapping_1 = {
                    'Slightly-infected': 'Early_Stage',
                    'Moderately-infected': 'Medium_Stage',
                    'Highly-infected': 'Severe_Stage',
                }
                severity_stage_1 = severity_mapping_1.get(Label_1, 'Undefined Stage')

                # Stress Label classification
                Label_G = classify_plant_stress(percentage_green)
                severity_mapping_G = {
                    'Low': 'Early_Stage',
                    'Medium': 'Medium_Stage',
                    'High': 'Severe_Stage',
                }
                severity_stage_G = severity_mapping_G.get(Label_G, 'Undefined Stage')

                hsv_image = cv2.cvtColor(image_2, cv2.COLOR_BGR2HSV)
                combined_mask = segment_spots(hsv_image)
                density = calculate_spot_density(combined_mask, hsv_image)

                low_threshold = 0.022
                moderate_threshold = 0.17
                Label_2 = 'Low' if density < low_threshold else ('Moderate' if density < moderate_threshold else 'High')
                severity_mapping = {
                    'Low': 'Early_Stage',
                    'Moderate': 'Medium_Stage',
                    'High': 'Severe_Stage',
                }
                severity_stage_2 = severity_mapping[Label_2]

                # Spots Distribution
                categories = ['Clustered', 'Scattered', 'Speckled', 'Varied']
                severity_levels_spots = {
                    'Scattered': 'Early_Stage',
                    'Speckled': 'Medium_Stage',
                    'Clustered': 'Severe_Stage',
                    'Varied': 'Severe_Stage'
                }
                prediction = model_2.predict(image, verbose=0)
                predicted_class_index = np.argmax(prediction)
                predicted_category = categories[predicted_class_index]
                severity_Spots = severity_levels_spots.get(predicted_category, 'Unknown')

                health_predictions.append(severity_stage_1)
                stress_predictions.append(severity_stage_G)
                density_predictions.append(severity_stage_2)
                spots_predictions.append(severity_Spots)
                texture_predictions.append(severity_texture)

            # Add actual labels
            health_labels.append(actual_stage)
            stress_labels.append(actual_stage)
            density_labels.append(actual_stage)
            spots_labels.append(actual_stage)
            texture_labels.append(actual_stage)
            actual_labels.append(actual_stage)

            final_prediction = majority_voting_1(severity_stage_1, severity_stage_G,
                                                 severity_stage_2, severity_Spots, severity_texture)
            final_predictions.append(final_prediction)

            if count >= 6000:
                break
        if count >= 6000:
            break

    # --- Original evaluation (printing) ---
    binary_class_labels = ['Healthy', 'Unhealthy']
    plot_Bconfusion_matrix(binary_actual_labels, binary_predictions, binary_class_labels, title='Healthy vs. Unhealthy Classification')
    Binary_accuracy = accuracy_score(binary_actual_labels, binary_predictions)
    print('\n')
    print(classification_report(binary_actual_labels, binary_predictions, target_names=binary_class_labels))
    binary_precision, binary_recall, binary_f1_score, _ = precision_recall_fscore_support(binary_actual_labels, binary_predictions, average='binary', pos_label='Unhealthy')
    print(f'Binary Classification Precision (Unhealthy): {binary_precision}')
    print(f'Binary Classification Recall (Unhealthy): {binary_recall}')
    print(f'Binary Classification F1-Score (Unhealthy): {binary_f1_score}')

    health_accuracy = accuracy_score(health_labels, health_predictions)
    print('\n\n Infection Level Confusion Matrix')
    class_labels = ['Healthy', 'Early_Stage', 'Medium_Stage', 'Severe_Stage']
    plot_confusion_matrix(health_labels, health_predictions, class_labels)
    print('\n')
    print(classification_report(health_labels, health_predictions, target_names=class_labels))
    health_precision, health_recall, health_f1_score, _ = precision_recall_fscore_support(health_labels, health_predictions, average='weighted')
    print(f'Infection Level Classification Precision: {health_precision}')
    print(f'Infection Level Classification Recall: {health_recall}')
    print(f'Infection Level Classification F1-Score: {health_f1_score}')

    print('\n\n Leaf Stress Confusion Matrix')
    stress_accuracy = accuracy_score(stress_labels, stress_predictions)
    plot_confusion_matrix(stress_labels, stress_predictions, class_labels)
    print('\n')
    print(classification_report(stress_labels, stress_predictions, target_names=class_labels))
    stress_precision, stress_recall, stress_f1_score, _ = precision_recall_fscore_support(stress_labels, stress_predictions, average='weighted')
    print(f'Leaf Stress Classification Precision: {stress_precision}')
    print(f'Leaf Stress Classification Recall: {stress_recall}')
    print(f'Leaf Stress Classification F1-Score: {stress_f1_score}')

    print('\n\n Spots Density Confusion Matrix')
    density_accuracy = accuracy_score(density_labels, density_predictions)
    plot_confusion_matrix(density_labels, density_predictions, class_labels)
    print('\n')
    print(classification_report(density_labels, density_predictions, target_names=class_labels))
    density_precision, density_recall, density_f1_score, _ = precision_recall_fscore_support(density_labels, density_predictions, average='weighted')
    print(f'Spots Density Classification Precision: {density_precision}')
    print(f'Spots Density Classification Recall: {density_recall}')
    print(f'Spots Density Classification F1-Score: {density_f1_score}')

    print('\n\n Texture Analysis Confusion Matrix')
    texture_accuracy = accuracy_score(texture_labels, texture_predictions)
    plot_confusion_matrix(texture_labels, texture_predictions, class_labels)
    print('\n')
    print(classification_report(texture_labels, texture_predictions, target_names=class_labels))
    texture_precision, texture_recall, texture_f1_score, _ = precision_recall_fscore_support(texture_labels, texture_predictions, average='weighted')
    print(f'Texture Analysis Classification Precision: {texture_precision}')
    print(f'Texture Analysis Classification Recall: {texture_recall}')
    print(f'Texture Analysis Classification F1-Score: {texture_f1_score}')

    print('\n\n Spots Distribution Confusion Matrix')
    spots_accuracy = accuracy_score(spots_labels, spots_predictions)
    plot_confusion_matrix(spots_labels, spots_predictions, class_labels)
    print('\n')
    print(classification_report(spots_labels, spots_predictions, target_names=class_labels))
    spots_precision, spots_recall, spots_f1_score, _ = precision_recall_fscore_support(spots_labels, spots_predictions, average='weighted')
    print(f'Spots Distribution Classification Precision: {spots_precision}')
    print(f'Spots Distribution Classification Recall: {spots_recall}')
    print(f'Spots Distribution Classification F1-Score: {spots_f1_score}')

    print('\n\n Majority Voting Confusion Matrix')
    final_accuracy = accuracy_score(actual_labels, final_predictions)
    plot_confusion_matrix(actual_labels, final_predictions, class_labels)
    print('\n')
    print(classification_report(actual_labels, final_predictions, target_names=class_labels))
    final_precision, final_recall, final_f1_score, _ = precision_recall_fscore_support(actual_labels, final_predictions, average='weighted')
    print(f'Majority Voting Classification Precision: {final_precision}')
    print(f'Majority Voting Classification Recall: {final_recall}')
    print(f'Majority Voting Classification F1-Score: {final_f1_score}')
    print('\n\n')

    print(f'Binary Classification Healthy_Unhealthy Accuracy: {Binary_accuracy}')
    print(f'Infection Level Accuracy: {health_accuracy}')
    print(f'Stress Classification Accuracy: {stress_accuracy}')
    print(f'Density Classification Accuracy: {density_accuracy}')
    print(f'Spots Distribution Accuracy: {spots_accuracy}')
    print(f'Texture Analysis Accuracy: {texture_accuracy}')
    print(f'Final Classification Accuracy: {final_accuracy}')
    print(f'Total Image: {count}')

    # Return all collected data for further analysis
    return {
        'binary_actual': binary_actual_labels,
        'binary_pred': binary_predictions,
        'health_labels': health_labels,
        'health_pred': health_predictions,
        'stress_labels': stress_labels,
        'stress_pred': stress_predictions,
        'density_labels': density_labels,
        'density_pred': density_predictions,
        'spots_labels': spots_labels,
        'spots_pred': spots_predictions,
        'texture_labels': texture_labels,
        'texture_pred': texture_predictions,
        'actual_labels': actual_labels,
        'final_pred': final_predictions,
        'total_count': count,
        'Binary_accuracy': Binary_accuracy,
        'health_accuracy': health_accuracy,
        'stress_accuracy': stress_accuracy,
        'density_accuracy': density_accuracy,
        'spots_accuracy': spots_accuracy,
        'texture_accuracy': texture_accuracy,
        'final_accuracy': final_accuracy
    }

# Specify the path to your image folder here
root_folder_path = f'{DATA_DIR}/All_Included_Final_Severity_Stage'
results = process_images_in_folder(root_folder_path)

## 10. Fold summaries of fixed predictions

This loop partitions already-computed predictions. It does not retrain the classifiers in each fold.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Extract predictions and true labels
all_preds = np.column_stack([
    results['health_pred'],
    results['stress_pred'],
    results['density_pred'],
    results['spots_pred'],
    results['texture_pred']
])
y_true = results['actual_labels']

# Encode string labels to integers
le = LabelEncoder()
y_enc = le.fit_transform(y_true)

# Weights (use the same order as columns)
weights = [94.13645, 94.81301, 94.43714, 84.11953, 96.84270]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_acc, cv_prec, cv_rec, cv_f1 = [], [], [], []

for train_idx, val_idx in skf.split(all_preds, y_enc):
    y_val = y_enc[val_idx]
    X_val = all_preds[val_idx]

    val_preds_enc = []
    for row in X_val:
        votes = {}
        for label_str, w in zip(row, weights):
            lbl = le.transform([label_str])[0]
            votes[lbl] = votes.get(lbl, 0) + w
        val_preds_enc.append(max(votes, key=votes.get))

    acc = accuracy_score(y_val, val_preds_enc)
    prec, rec, f1, _ = precision_recall_fscore_support(y_val, val_preds_enc, average='weighted')

    cv_acc.append(acc)
    cv_prec.append(prec)
    cv_rec.append(rec)
    cv_f1.append(f1)

print(f"CV Overall Accuracy:  {np.mean(cv_acc):.4f} ± {np.std(cv_acc):.4f}")
print(f"CV Overall Precision: {np.mean(cv_prec):.4f} ± {np.std(cv_prec):.4f}")
print(f"CV Overall Recall:    {np.mean(cv_rec):.4f} ± {np.std(cv_rec):.4f}")
print(f"CV Overall F1-score:  {np.mean(cv_f1):.4f} ± {np.std(cv_f1):.4f}")

## 11. Attribute ablations on fixed predictions

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np
import pandas as pd

# Assume `results` DataFrame and `le` (LabelEncoder) are already defined as in your code
# all_preds: shape (n_samples, 5) with columns: health, stress, density, spots, texture
# y_enc: encoded true labels
# weights: original weights for each attribute (order as above)

# Store ablation results
ablation_results = {}

# Loop over each attribute index to exclude (0 to 4)
for exclude_idx in range(5):
    # Create modified weights (set excluded weight to 0)
    modified_weights = weights.copy()
    modified_weights[exclude_idx] = 0

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_acc, cv_prec, cv_rec, cv_f1 = [], [], [], []

    for train_idx, val_idx in skf.split(all_preds, y_enc):
        y_val = y_enc[val_idx]
        X_val = all_preds[val_idx]

        val_preds_enc = []
        for row in X_val:
            votes = {}
            for i, (label_str, w) in enumerate(zip(row, modified_weights)):
                if w == 0:   # skip excluded attribute
                    continue
                lbl = le.transform([label_str])[0]
                votes[lbl] = votes.get(lbl, 0) + w
            # If all weights are zero (should not happen), handle edge case
            if not votes:
                # fallback: pick most frequent class in training? Here we simply assign a default (e.g., 0)
                val_preds_enc.append(0)
            else:
                val_preds_enc.append(max(votes, key=votes.get))

        acc = accuracy_score(y_val, val_preds_enc)
        prec, rec, f1, _ = precision_recall_fscore_support(y_val, val_preds_enc, average='weighted')

        cv_acc.append(acc)
        cv_prec.append(prec)
        cv_rec.append(rec)
        cv_f1.append(f1)

    # Store mean ± std for this ablation
    ablation_results[f"exclude_{exclude_idx}"] = {
        'accuracy': (np.mean(cv_acc), np.std(cv_acc)),
        'precision': (np.mean(cv_prec), np.std(cv_prec)),
        'recall': (np.mean(cv_rec), np.std(cv_rec)),
        'f1': (np.mean(cv_f1), np.std(cv_f1))
    }

# Print ablation results
print("Ablation Study (Excluding One Attribute):")
for excl, metrics in ablation_results.items():
    print(f"\n{excl}:")
    for metric, (mean_val, std_val) in metrics.items():
        print(f"  {metric}: {mean_val:.4f} ± {std_val:.4f}")

## 12. Timing experiment

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random
import time  # <--- ADDED for timing
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

# Assumed predefined functions (you must have them defined earlier in the notebook):
# extract_foreground, preprocess_shiny, calculate_green_peak, threshold_green, separate_regions,
# calculate_area_percentages, classify_plant_health, classify_plant_stress, segment_spots, calculate_spot_density,
# load_and_resize_image, preprocess_input, model, model_2, model_3, scaler, analyze_texture, majority_voting_1,
# plot_Bconfusion_matrix, plot_confusion_matrix

def process_images_in_folder(root_folder_path):
    subfolders = [os.path.join(root_folder_path, f) for f in os.listdir(root_folder_path) if os.path.isdir(os.path.join(root_folder_path, f))]

    # ... (all your existing list initializations) ...
    health_labels = []
    health_predictions = []
    stress_labels = []
    stress_predictions = []
    density_labels = []
    density_predictions = []
    spots_labels = []
    spots_predictions = []
    texture_labels = []
    texture_predictions = []
    final_predictions = []
    actual_labels = []
    count = 0
    binary_predictions = []
    binary_actual_labels = []
    i = 0

    # --- NEW: timing accumulators ---
    load_times = []
    health_times = []
    green_times = []
    density_times = []
    spots_times = []
    texture_times = []
    vote_times = []
    total_times = []

    for subfolder in subfolders:
        actual_stage = os.path.basename(subfolder)
        actual_binary_label = 'Healthy' if actual_stage == 'Healthy' else 'Unhealthy'
        files = [f for f in os.listdir(subfolder) if os.path.isfile(os.path.join(subfolder, f))]
        image_files = [f for f in files if f.lower().endswith(('.png', '.jpg','.JPG', '.jpeg'))]

        for image_file in image_files:
            image_path = os.path.join(subfolder, image_file)
            count += 1
            i += 1
            print(i)

            # --- Total time start ---
            t_total_start = time.time()

            # --- 1. Load and resize image ---
            t0 = time.time()
            im = cv2.imread(image_path)
            image = load_and_resize_image(im)
            image_2 = image.copy()
            load_times.append(time.time() - t0)

            # --- 2. Health model prediction ---
            t0 = time.time()
            image_batch = np.expand_dims(image, axis=0)
            image_batch = preprocess_input(image_batch)
            prediction = model.predict(image_batch, verbose=0)
            health_times.append(time.time() - t0)

            threshold = 0.5
            predicted_class = 'Healthy' if prediction[0][0] < threshold else 'Unhealthy'

            binary_actual_labels.append(actual_binary_label)
            binary_predictions.append(predicted_class)

            if predicted_class == 'Healthy':
                # For healthy images, set all severity stages to 'Healthy' so voting works
                severity_stage_1 = severity_stage_G = severity_stage_2 = severity_Spots = severity_texture = 'Healthy'
                health_predictions.append('Healthy')
                stress_predictions.append('Healthy')
                density_predictions.append('Healthy')
                spots_predictions.append('Healthy')
                texture_predictions.append('Healthy')
                # Record zero time for skipped steps
                green_times.append(0)
                density_times.append(0)
                spots_times.append(0)
                texture_times.append(0)
            else:
                # --- 3. Greenness / Infection Level (foreground, green peak, etc.) ---
                t0 = time.time()
                foreground, foreground_mask = extract_foreground(image_2)
                preprocessed_foreground = preprocess_shiny(foreground)

                green_peak = calculate_green_peak(preprocessed_foreground)
                green_mask = threshold_green(foreground, green_peak)
                green_region, non_green_region = separate_regions(foreground, green_mask)

                percentage_green, percentage_non_green, green_area, non_green_area = calculate_area_percentages(green_mask, non_green_region, foreground_mask)

                # Infection Level classification
                Label_1 = classify_plant_health(percentage_non_green)
                severity_mapping_1 = {
                    'Slightly-infected': 'Early_Stage',
                    'Moderately-infected': 'Medium_Stage',
                    'Highly-infected': 'Severe_Stage',
                }
                severity_stage_1 = severity_mapping_1.get(Label_1, 'Undefined Stage')

                # Stress Level classification
                Label_G = classify_plant_stress(percentage_green)
                severity_mapping_G = {
                    'Low': 'Early_Stage',
                    'Medium': 'Medium_Stage',
                    'High': 'Severe_Stage',
                }
                severity_stage_G = severity_mapping_G.get(Label_G, 'Undefined Stage')
                green_times.append(time.time() - t0)

                # --- 4. Spot Density ---
                t0 = time.time()
                hsv_image = cv2.cvtColor(image_2, cv2.COLOR_BGR2HSV)
                combined_mask = segment_spots(hsv_image)
                density = calculate_spot_density(combined_mask, hsv_image)

                low_threshold = 0.022
                moderate_threshold = 0.17
                Label_2 = 'Low' if density < low_threshold else ('Moderate' if density < moderate_threshold else 'High')
                severity_mapping = {
                    'Low': 'Early_Stage',
                    'Moderate': 'Medium_Stage',
                    'High': 'Severe_Stage',
                }
                severity_stage_2 = severity_mapping[Label_2]
                density_times.append(time.time() - t0)

                # --- 5. Spots Distribution ---
                t0 = time.time()
                categories = ['Clustered', 'Scattered', 'Speckled', 'Varied']
                severity_levels_spots = {
                    'Scattered': 'Early_Stage',
                    'Speckled': 'Medium_Stage',
                    'Clustered': 'Severe_Stage',
                    'Varied': 'Severe_Stage'
                }
                # model_2 expects the same preprocessed image as the health model? We'll reuse image_batch.
                # If model_2 requires a different preprocessing, adjust accordingly.
                spots_pred = model_2.predict(image_batch, verbose=0)
                predicted_class_index = np.argmax(spots_pred)
                predicted_category = categories[predicted_class_index]
                severity_Spots = severity_levels_spots.get(predicted_category, 'Unknown')
                spots_times.append(time.time() - t0)

                # --- 6. Texture Analysis ---
                t0 = time.time()
                if non_green_region.ndim == 3:
                    image3 = cv2.cvtColor(non_green_region, cv2.COLOR_BGR2GRAY)
                else:
                    image3 = non_green_region
                results = analyze_texture(image3)
                stage_mapping = {0:'Early_Stage', 1: 'Medium_Stage', 2: 'Severe_Stage'}
                features_array = np.array(list(results.values())).reshape(1, -1)
                scaled_features = scaler.transform(features_array)
                numeric_prediction = model_3.predict(scaled_features)
                severity_texture = stage_mapping[numeric_prediction[0]]
                texture_times.append(time.time() - t0)

                # Append predictions for evaluation
                health_predictions.append(severity_stage_1)
                stress_predictions.append(severity_stage_G)
                density_predictions.append(severity_stage_2)
                spots_predictions.append(severity_Spots)
                texture_predictions.append(severity_texture)

            # --- 7. Final Voting (ensemble) ---
            t0 = time.time()
            final_prediction = majority_voting_1(severity_stage_1, severity_stage_G,
                                                 severity_stage_2, severity_Spots, severity_texture)
            vote_times.append(time.time() - t0)

            # Store actual labels and final prediction
            health_labels.append(actual_stage)
            stress_labels.append(actual_stage)
            density_labels.append(actual_stage)
            spots_labels.append(actual_stage)
            texture_labels.append(actual_stage)
            actual_labels.append(actual_stage)
            final_predictions.append(final_prediction)

            # Total time for this image
            total_times.append(time.time() - t_total_start)

            if count >= 6000:
                break
        if count >= 6000:
            break

    # --- Compute and print average inference times (in milliseconds) ---
    def avg_ms(t_list):
        return np.mean(t_list) * 1000 if t_list else 0

    print("\n=== INFERENCE TIME PER IMAGE (average over {} images) ===".format(len(total_times)))
    print(f"Load & resize:         {avg_ms(load_times):.3f} ms")
    print(f"Health model:          {avg_ms(health_times):.3f} ms")
    print(f"Greenness/Infection:   {avg_ms(green_times):.3f} ms")
    print(f"Spot density:          {avg_ms(density_times):.3f} ms")
    print(f"Spots distribution:    {avg_ms(spots_times):.3f} ms")
    print(f"Texture analysis:      {avg_ms(texture_times):.3f} ms")
    print(f"Ensemble voting:       {avg_ms(vote_times):.3f} ms")
    print(f"----------------------------------------")
    print(f"Total per image:       {avg_ms(total_times):.3f} ms\n")

    # --- Original evaluation (printing) ---
    binary_class_labels = ['Healthy', 'Unhealthy']
    plot_Bconfusion_matrix(binary_actual_labels, binary_predictions, binary_class_labels, title='Healthy vs. Unhealthy Classification')
    Binary_accuracy = accuracy_score(binary_actual_labels, binary_predictions)
    print('\n')
    print(classification_report(binary_actual_labels, binary_predictions, target_names=binary_class_labels))
    binary_precision, binary_recall, binary_f1_score, _ = precision_recall_fscore_support(binary_actual_labels, binary_predictions, average='binary', pos_label='Unhealthy')
    print(f'Binary Classification Precision (Unhealthy): {binary_precision}')
    print(f'Binary Classification Recall (Unhealthy): {binary_recall}')
    print(f'Binary Classification F1-Score (Unhealthy): {binary_f1_score}')

    health_accuracy = accuracy_score(health_labels, health_predictions)
    print('\n\n Infection Level Confusion Matrix')
    class_labels = ['Healthy', 'Early_Stage', 'Medium_Stage', 'Severe_Stage']
    plot_confusion_matrix(health_labels, health_predictions, class_labels)
    print('\n')
    print(classification_report(health_labels, health_predictions, target_names=class_labels))
    health_precision, health_recall, health_f1_score, _ = precision_recall_fscore_support(health_labels, health_predictions, average='weighted')
    print(f'Infection Level Classification Precision: {health_precision}')
    print(f'Infection Level Classification Recall: {health_recall}')
    print(f'Infection Level Classification F1-Score: {health_f1_score}')

    print('\n\n Leaf Stress Confusion Matrix')
    stress_accuracy = accuracy_score(stress_labels, stress_predictions)
    plot_confusion_matrix(stress_labels, stress_predictions, class_labels)
    print('\n')
    print(classification_report(stress_labels, stress_predictions, target_names=class_labels))
    stress_precision, stress_recall, stress_f1_score, _ = precision_recall_fscore_support(stress_labels, stress_predictions, average='weighted')
    print(f'Leaf Stress Classification Precision: {stress_precision}')
    print(f'Leaf Stress Classification Recall: {stress_recall}')
    print(f'Leaf Stress Classification F1-Score: {stress_f1_score}')

    print('\n\n Spots Density Confusion Matrix')
    density_accuracy = accuracy_score(density_labels, density_predictions)
    plot_confusion_matrix(density_labels, density_predictions, class_labels)
    print('\n')
    print(classification_report(density_labels, density_predictions, target_names=class_labels))
    density_precision, density_recall, density_f1_score, _ = precision_recall_fscore_support(density_labels, density_predictions, average='weighted')
    print(f'Spots Density Classification Precision: {density_precision}')
    print(f'Spots Density Classification Recall: {density_recall}')
    print(f'Spots Density Classification F1-Score: {density_f1_score}')

    print('\n\n Texture Analysis Confusion Matrix')
    texture_accuracy = accuracy_score(texture_labels, texture_predictions)
    plot_confusion_matrix(texture_labels, texture_predictions, class_labels)
    print('\n')
    print(classification_report(texture_labels, texture_predictions, target_names=class_labels))
    texture_precision, texture_recall, texture_f1_score, _ = precision_recall_fscore_support(texture_labels, texture_predictions, average='weighted')
    print(f'Texture Analysis Classification Precision: {texture_precision}')
    print(f'Texture Analysis Classification Recall: {texture_recall}')
    print(f'Texture Analysis Classification F1-Score: {texture_f1_score}')

    print('\n\n Spots Distribution Confusion Matrix')
    spots_accuracy = accuracy_score(spots_labels, spots_predictions)
    plot_confusion_matrix(spots_labels, spots_predictions, class_labels)
    print('\n')
    print(classification_report(spots_labels, spots_predictions, target_names=class_labels))
    spots_precision, spots_recall, spots_f1_score, _ = precision_recall_fscore_support(spots_labels, spots_predictions, average='weighted')
    print(f'Spots Distribution Classification Precision: {spots_precision}')
    print(f'Spots Distribution Classification Recall: {spots_recall}')
    print(f'Spots Distribution Classification F1-Score: {spots_f1_score}')

    print('\n\n Majority Voting Confusion Matrix')
    final_accuracy = accuracy_score(actual_labels, final_predictions)
    plot_confusion_matrix(actual_labels, final_predictions, class_labels)
    print('\n')
    print(classification_report(actual_labels, final_predictions, target_names=class_labels))
    final_precision, final_recall, final_f1_score, _ = precision_recall_fscore_support(actual_labels, final_predictions, average='weighted')
    print(f'Majority Voting Classification Precision: {final_precision}')
    print(f'Majority Voting Classification Recall: {final_recall}')
    print(f'Majority Voting Classification F1-Score: {final_f1_score}')
    print('\n\n')

    print(f'Binary Classification Healthy_Unhealthy Accuracy: {Binary_accuracy}')
    print(f'Infection Level Accuracy: {health_accuracy}')
    print(f'Stress Classification Accuracy: {stress_accuracy}')
    print(f'Density Classification Accuracy: {density_accuracy}')
    print(f'Spots Distribution Accuracy: {spots_accuracy}')
    print(f'Texture Analysis Accuracy: {texture_accuracy}')
    print(f'Final Classification Accuracy: {final_accuracy}')
    print(f'Total Image: {count}')

    # Return all collected data for further analysis
    return {
        'binary_actual': binary_actual_labels,
        'binary_pred': binary_predictions,
        'health_labels': health_labels,
        'health_pred': health_predictions,
        'stress_labels': stress_labels,
        'stress_pred': stress_predictions,
        'density_labels': density_labels,
        'density_pred': density_predictions,
        'spots_labels': spots_labels,
        'spots_pred': spots_predictions,
        'texture_labels': texture_labels,
        'texture_pred': texture_predictions,
        'actual_labels': actual_labels,
        'final_pred': final_predictions,
        'total_count': count,
        'Binary_accuracy': Binary_accuracy,
        'health_accuracy': health_accuracy,
        'stress_accuracy': stress_accuracy,
        'density_accuracy': density_accuracy,
        'spots_accuracy': spots_accuracy,
        'texture_accuracy': texture_accuracy,
        'final_accuracy': final_accuracy,
        # Optionally return timing data
        'timing': {
            'load_ms': avg_ms(load_times),
            'health_ms': avg_ms(health_times),
            'green_ms': avg_ms(green_times),
            'density_ms': avg_ms(density_times),
            'spots_ms': avg_ms(spots_times),
            'texture_ms': avg_ms(texture_times),
            'vote_ms': avg_ms(vote_times),
            'total_ms': avg_ms(total_times)
        }
    }

# Specify the path to your image folder here
root_folder_path = f'{DATA_DIR}/All_Included_Final_Severity_Stage'
results = process_images_in_folder(root_folder_path)

## 13. Alternative weighted-voting helper

This alternative uses a different texture weight. Repeated predicted labels overwrite earlier weights in its dictionary. Confirm the publication-generating variant before presenting this as the definitive implementation.

In [ ]:
def majority_voting_1(health_pred, stress_pred, density_pred, spots_pred, texture_pred):
    # Define the weights based on model accuracy.
    weights = {
        'health_pred': 94.13645,
        'stress_pred': 94.81301,
        'density_pred': 94.43714,
        'spots_pred': 84.11953,
        'texture_pred': 97.14
    }

    predictions = [health_pred, stress_pred, density_pred, spots_pred, texture_pred]

    # If all predictions are the same, return that prediction.
    if len(set(predictions)) == 1:
        return health_pred

    # Associate each prediction with its respective weight
    weighted_predictions = {
        health_pred: weights['health_pred'],
        stress_pred: weights['stress_pred'],
        density_pred: weights['density_pred'],
        spots_pred: weights['spots_pred'],
        texture_pred: weights['texture_pred']
    }

    # Calculate the total weight for each unique prediction
    prediction_scores = {}
    for pred, weight in weighted_predictions.items():
        if pred in prediction_scores:
            prediction_scores[pred] += weight
        else:
            prediction_scores[pred] = weight

    # Sort by weighted score and return the prediction with the highest score
    sorted_weighted_predictions = sorted(prediction_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_weighted_predictions[0][0]


## 14. Evaluate the alternative helper

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

# Assumed predefined functions:
# extract_foreground, preprocess_shiny, calculate_green_peak, threshold_green, separate_regions,
# calculate_area_percentages, classify_plant_health, classify_plant_stress, segment_spots, calculate_spot_density,
# load_and_resize_image, preprocess_input, model

def process_images_in_folder(root_folder_path):
    subfolders = [os.path.join(root_folder_path, f) for f in os.listdir(root_folder_path) if os.path.isdir(os.path.join(root_folder_path, f))]
    health_labels = []
    health_predictions = []
    stress_labels = []
    stress_predictions = []
    density_labels = []
    density_predictions = []
    spots_labels = []
    spots_predictions = []
    texture_labels = []
    texture_predictions = []
    final_predictions = []  # Store final predictions after majority voting
    actual_labels = []  # Store actual labels for accuracy calculation
    count = 0
    # Variables for binary classification (Healthy vs. Unhealthy)
    binary_predictions = []
    binary_actual_labels = []
    i=0


    for subfolder in subfolders:
        actual_stage = os.path.basename(subfolder)  # The actual stage is the subfolder name
        if actual_stage == 'Healthy':
           actual_binary_label = 'Healthy'
        else:
          actual_binary_label = 'Unhealthy'

        files = [f for f in os.listdir(subfolder) if os.path.isfile(os.path.join(subfolder, f))]
        image_files = [f for f in files if f.lower().endswith(('.png', '.jpg','.JPG', '.jpeg'))]

        for image_file in image_files:
            image_path = os.path.join(subfolder, image_file)
            count += 1
            i+=1
            print(i)

            im = cv2.imread(image_path)
            image = load_and_resize_image(im)
            image_2 = image.copy()
            image = np.expand_dims(image, axis=0)
            image = preprocess_input(image)
            prediction = model.predict(image)
            threshold = 0.5
            predicted_class = 'Healthy' if prediction[0][0] < threshold else 'Unhealthy'
                    # Record binary predictions and actual labels

            binary_actual_labels.append(actual_binary_label)
            binary_predictions.append(predicted_class)


            if predicted_class == 'Healthy':
                severity_mapping_1 = 'Healthy'
                severity_stage_G = 'Healthy'
                severity_stage_2 = 'Healthy'
                severity_levels_spots='Healthy'
                severity_texture='Healthy'
                # Assuming that 'NULL' maps to 'Healthy'
                health_predictions.append(severity_mapping_1)  # Or 'Healthy' if you have such a label
                stress_predictions.append(severity_stage_G )  # Same assumption
                density_predictions.append(severity_stage_2)  # Same assumption
                spots_predictions.append(severity_levels_spots)  # Same assumption
                texture_predictions.append(severity_texture)  # Same assumption



            else:
                foreground, foreground_mask = extract_foreground(image_2)
                preprocessed_foreground = preprocess_shiny(foreground)

                # Calculate the green color peak on the foreground
                green_peak = calculate_green_peak(preprocessed_foreground)
                # Set the green threshold and obtain the green mask for the foreground
                green_mask = threshold_green(foreground, green_peak)

                # Separate and refine the green and non-green regions from the foreground
                green_region, non_green_region = separate_regions(foreground, green_mask)

                # Calculate area percentages
                percentage_green, percentage_non_green, green_area, non_green_area = calculate_area_percentages(green_mask, non_green_region, foreground_mask)

                if non_green_region.ndim == 3:
                  image3 = cv2.cvtColor(non_green_region, cv2.COLOR_BGR2GRAY)

                # Analyze texture
                results = analyze_texture(image3)
                # Define the mapping from numerical prediction to categorical label
                stage_mapping = {0:'Early_Stage', 1: 'Medium_Stage', 2: 'Severe_Stage'}
                # Convert the dictionary of features to an array-like structure
                features_array = np.array(list(results.values())).reshape(1, -1)
                # Normalize the features using the loaded scaler
                scaled_features = scaler.transform(features_array)
                #print(scaled_features.shape)  # Check the current shape
                # Assuming your model expects input of shape (1, height, width, channels)
                # You may need to reshape or adjust your input here.

                # Use the model to predict using the normalized features
                # Use the model to predict using the normalized features
                numeric_prediction = model_3.predict(scaled_features)

                # Map the numerical prediction to a categorical label
                texture_prediction = stage_mapping[numeric_prediction[0]]
                #print(texture_prediction)
                severity_texture=texture_prediction




                 #Infection Label classification

                Label_1 = classify_plant_health(percentage_non_green)
                severity_mapping_1 = {
                    'Slightly-infected': 'Early_Stage',
                    'Moderately-infected': 'Medium_Stage',
                    'Highly-infected': 'Severe_Stage',
                }
                severity_stage_1 = severity_mapping_1.get(Label_1, 'Undefined Stage')

                #Stress Label classification
                Label_G = classify_plant_stress(percentage_green)
                severity_mapping_G = {
                    'Low': 'Early_Stage',
                    'Medium': 'Medium_Stage',
                    'High': 'Severe_Stage',
                }
                severity_stage_G = severity_mapping_G.get(Label_G, 'Undefined Stage')

                hsv_image = cv2.cvtColor(image_2, cv2.COLOR_BGR2HSV)
                combined_mask = segment_spots(hsv_image)
                density = calculate_spot_density(combined_mask, hsv_image)

                # Define thresholds for density labeling
                low_threshold = 0.022  # Placeholder value for low density->0.023
                moderate_threshold = 0.17  # Placeholder value for moderate density ->0.17
                Label_2 = 'Low' if density < low_threshold else ('Moderate' if density < moderate_threshold else 'High')

                severity_mapping = {
                    'Low': 'Early_Stage',
                    'Moderate': 'Medium_Stage',
                    'High': 'Severe_Stage',
                }

                severity_stage_2 = severity_mapping[Label_2]

                # Define categories
                #Check Spots Distribution
                categories = ['Clustered', 'Scattered', 'Speckled', 'Varied']

                # Define severity levels
                severity_levels_spots = {
                    'Scattered': 'Early_Stage',
                    'Speckled': 'Medium_Stage',
                    'Clustered': 'Severe_Stage',
                    'Varied': 'Severe_Stage'
                }
                # Make prediction
                prediction = model_2.predict(image)

                # Convert probabilities to class label
                predicted_class_index = np.argmax(prediction)
                predicted_category = categories[predicted_class_index]

                # Map severity
                severity_Spots = severity_levels_spots.get(predicted_category, 'Unknown')

            # Check if severity stages are the same and increment the counter
                health_predictions.append(severity_stage_1)
                stress_predictions.append(severity_stage_G)
                density_predictions.append(severity_stage_2)
                spots_predictions.append(severity_Spots)
                texture_predictions.append(texture_prediction)

            # Add actual labels for accuracy calculation
            health_labels.append(actual_stage)
            stress_labels.append(actual_stage)
            density_labels.append(actual_stage)
            spots_labels.append(actual_stage)
            texture_labels.append(actual_stage)
            actual_labels.append(actual_stage)


            # Majority voting to determine final prediction
            final_prediction = majority_voting_2(severity_stage_1, severity_stage_G, severity_stage_2,severity_Spots,severity_texture)
            final_predictions.append(final_prediction)



            if count >= 6000:  # Stopping condition
                break

        if count >= 6000:
            break

    # Binary classification labels
    binary_class_labels = ['Healthy', 'Unhealthy']
    # Plotting confusion matrix for binary classification
    plot_Bconfusion_matrix(binary_actual_labels, binary_predictions, binary_class_labels, title='Healthy vs. Unhealthy Classification')
    Binary_accuracy = accuracy_score(binary_actual_labels , binary_predictions)
    # Calculate precision, recall, and F1-score for binary classification
    binary_precision, binary_recall, binary_f1_score, _ = precision_recall_fscore_support(binary_actual_labels, binary_predictions, average='binary', pos_label='Unhealthy')
    print('\n')
    print(classification_report(binary_actual_labels, binary_predictions, target_names=binary_class_labels))

    # Print precision, recall, and F1-score for binary classification
    print(f'Binary Classification Precision (Unhealthy): {binary_precision}')
    print(f'Binary Classification Recall (Unhealthy): {binary_recall}')
    print(f'Binary Classification F1-Score (Unhealthy): {binary_f1_score}')


    # Accuracy calculations
    health_accuracy = accuracy_score(health_labels, health_predictions)
    print('\n\n Infection Level Confusion Matrix')
    class_labels = ['Healthy','Early_Stage', 'Medium_Stage', 'Severe_Stage']  # Adjust labels as necessary
    plot_confusion_matrix(health_labels, health_predictions, class_labels)
    print('\n')
    print(classification_report(health_labels, health_predictions, target_names=class_labels))
    # Calculate precision, recall, and F1-score for Infection Level Classification
    health_precision, health_recall, health_f1_score, _ = precision_recall_fscore_support(health_labels, health_predictions, average='weighted')

    # Print out precision, recall, and F1-score for Infection Level Classification
    print(f'Infection Level Classification Precision: {health_precision}')
    print(f'Infection Level Classification Recall: {health_recall}')
    print(f'Infection Level Classification F1-Score: {health_f1_score}')

    # Then follow with your accuracy calculations and printing
    print('\n\n Leaf Stress Confusion Matrix')
    stress_accuracy = accuracy_score(stress_labels, stress_predictions)
    plot_confusion_matrix(stress_labels, stress_predictions, class_labels)
    print('\n')
    print(classification_report(stress_labels, stress_predictions, target_names=class_labels))
    # Calculate precision, recall, and F1-score for Leaf Stress Classification
    stress_precision, stress_recall, stress_f1_score, _ = precision_recall_fscore_support(stress_labels, stress_predictions, average='weighted')

    # Print precision, recall, and F1-score for Leaf Stress Classification
    print(f'Leaf Stress Classification Precision: {stress_precision}')
    print(f'Leaf Stress Classification Recall: {stress_recall}')
    print(f'Leaf Stress Classification F1-Score: {stress_f1_score}')

    print('\n\n Spots Density Confusion Matrix')
    density_accuracy = accuracy_score(density_labels, density_predictions)
    plot_confusion_matrix(density_labels, density_predictions, class_labels)
    print('\n')
    print(classification_report(density_labels, density_predictions, target_names=class_labels))

    # Calculate precision, recall, and F1-score for Spots Density Classification
    density_precision, density_recall, density_f1_score, _ = precision_recall_fscore_support(density_labels, density_predictions, average='weighted')

    # Print out precision, recall, and F1-score for Spots Density Classification
    print(f'Spots Density Classification Precision: {density_precision}')
    print(f'Spots Density Classification Recall: {density_recall}')
    print(f'Spots Density Classification F1-Score: {density_f1_score}')

    print('\n\n Texture Analysis Confusion Matrix')
    texture_accuracy = accuracy_score(texture_labels, texture_predictions)
    plot_confusion_matrix(texture_labels, texture_predictions, class_labels)
    print('\n')
    print(classification_report(texture_labels, texture_predictions, target_names=class_labels))
    # Calculate precision, recall, and F1-score for Texture Analysis Classification
    texture_precision, texture_recall, texture_f1_score, _ = precision_recall_fscore_support(texture_labels, texture_predictions, average='weighted')

    # Print precision, recall, and F1-score for Texture Analysis Classification
    print(f'Texture Analysis Classification Precision: {texture_precision}')
    print(f'Texture Analysis Classification Recall: {texture_recall}')
    print(f'Texture Analysis Classification F1-Score: {texture_f1_score}')

    print('\n\n Spots Distribution Confusion Matrix')
    spots_accuracy = accuracy_score(spots_labels, spots_predictions)
    plot_confusion_matrix(spots_labels, spots_predictions, class_labels)
    print('\n')
    print(classification_report(spots_labels, spots_predictions, target_names=class_labels))
    # Calculate precision, recall, and F1-score for Spots Distribution Classification
    spots_precision, spots_recall, spots_f1_score, _ = precision_recall_fscore_support(spots_labels, spots_predictions, average='weighted')

    # Print out precision, recall, and F1-score for Spots Distribution Classification
    print(f'Spots Distribution Classification Precision: {spots_precision}')
    print(f'Spots Distribution Classification Recall: {spots_recall}')
    print(f'Spots Distribution Classification F1-Score: {spots_f1_score}')

    print('\n\n Majority Voting Confusion Matrix')
    # Calculate and print accuracy
    final_accuracy = accuracy_score(actual_labels, final_predictions)
    plot_confusion_matrix(actual_labels, final_predictions, class_labels)
    print('\n')
    print(classification_report(actual_labels, final_predictions, target_names=class_labels))
    # Calculate precision, recall, and F1-score for Majority Voting Classification
    final_precision, final_recall, final_f1_score, _ = precision_recall_fscore_support(actual_labels, final_predictions, average='weighted')

    # Print out precision, recall, and F1-score for Majority Voting Classification
    print(f'Majority Voting Classification Precision: {final_precision}')
    print(f'Majority Voting Classification Recall: {final_recall}')
    print(f'Majority Voting Classification F1-Score: {final_f1_score}')
    print('\n\n')

    # Visualization call (add this)

    print(f'Binary Classification Healthy_Unhealthy Accuracy: {Binary_accuracy}')
    print(f'Infection Level Accuracy: {health_accuracy}')
    print(f'Stress Classification Accuracy: {stress_accuracy}')
    print(f'Density Classification Accuracy: {density_accuracy}')
    print(f'Spots Distribution Accuracy: {spots_accuracy}')
    print(f'Texture Analysis Accuracy: {texture_accuracy}')
    print(f'Final Classification Accuracy: {final_accuracy}')
    print(f'Total Image: {count}')

    return {
        'total_images_processed': count,
        'health_accuracy': health_accuracy,
        'stress_accuracy': stress_accuracy,
        'density_accuracy': density_accuracy,
        'spots_accuracy': spots_accuracy,
        'texture_accuracy': texture_accuracy,
        'final_accuracy': final_accuracy
    }
# Specify the path to your image folder here
root_folder_path = f'{DATA_DIR}/All_Included_Final_Severity_Stage'
results = process_images_in_folder(root_folder_path)


## 15. Exploratory per-stage routines

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Main function to process the images in the folder
def process_images_in_folder(folder_path):
    # List all image files in the folder
    files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
    image_files = [f for f in files if f.lower().endswith(('.png','.JPG', '.jpg', '.jpeg'))]

    #Shuffle the list of image files to process them in random order
    #random.shuffle(image_files)
    match_counter=0
    WH=0
    count=0
    Total_match=0

    # Process each image file
    for image_file in image_files:
        image_path = os.path.join(folder_path, image_file)
        count = count + 1
        im = cv2.imread(image_path)
        image = load_and_resize_image(im)
        image_2 = image  # Create a copy of the original image
        # Preprocess the image
        image = np.expand_dims(image, axis=0)  # Add a batch dimension
        image = preprocess_input(image)
        # Predict the class of the image
        prediction = model.predict(image)

        # Threshold the prediction (if your model uses a sigmoid activation in the last layer)
        threshold = 0.5
        predicted_class = 'Healthy' if prediction[0][0] < threshold else 'Unhealthy'

        if predicted_class == 'Healthy':  # If the model predicts 'Healthy'
            Label_1 = 'Non-infected'
            Label_2 = 'Spotless'
            Label_G=  'No-Stress'
            severity_stage = 'NULL'
            Label_texture='NULL'
            WH +=1
            rgb_image = cv2.cvtColor(image_2, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(2, 2))
            plt.imshow(rgb_image)
            plt.title(f'{image_file}\n\nInfection Level : {Label_1}\n\nStress Level: {Label_G}\n\nSpots Density: {Label_2}\n\nSeverity:{severity_stage}\n\nTexture: {Label_texture}')

            plt.show()
        else:  # If the model predicts 'Unhealthy'
            foreground, foreground_mask = extract_foreground(image_2)
            preprocessed_foreground = preprocess_shiny(foreground)

            # Calculate the green color peak on the foreground
            green_peak = calculate_green_peak(preprocessed_foreground)
            # Set the green threshold and obtain the green mask for the foreground
            green_mask = threshold_green(foreground, green_peak)

            # Separate and refine the green and non-green regions from the foreground
            green_region, non_green_region = separate_regions(foreground, green_mask)

            # Calculate area percentages
            percentage_green, percentage_non_green, green_area, non_green_area = calculate_area_percentages(green_mask, non_green_region, foreground_mask)

            print(f'Processing image # {count} & Name :{image_file}')
            print(f"Percentage of Green Area: {percentage_green:.2f}%")
            print(f"Percentage of Non-Green Area: {percentage_non_green:.2f}%")


            # Label classification
            Label_1 = classify_plant_health(percentage_non_green)
            severity_mapping_1 = {
                'Slightly-infected': 'Early Stage',
                'Moderately-infected': 'Medium Stage',
                'Highly-infected': 'Severe Stage',
            }
            severity_stage_1 = severity_mapping_1.get(Label_1, 'Undefined Stage')

                        # Label classification
            Label_G = classify_plant_stress(percentage_green)
            severity_mapping_G = {
                'Low': 'Early Stage',
                'Medium': 'Medium Stage',
                'High': 'Severe Stage',
            }
            severity_stage_G = severity_mapping_G.get(Label_G, 'Undefined Stage')

            if non_green_region.ndim == 3:
              image3 = cv2.cvtColor(non_green_region, cv2.COLOR_BGR2GRAY)

                # Analyze texture
            results = analyze_texture(image3)
                # Define the mapping from numerical prediction to categorical label
            stage_mapping = {0:'Early_Stage', 1: 'Medium_Stage', 2: 'Severe_Stage'}

                # Convert the dictionary of features to an array-like structure
            features_array = np.array(list(results.values())).reshape(1, -1)
                # Normalize the features using the loaded scaler
            scaled_features = scaler.transform(features_array)
                #print(scaled_features.shape)  # Check the current shape
                # Assuming your model expects input of shape (1, height, width, channels)
                # You may need to reshape or adjust your input here.

                # Use the model to predict using the normalized features
                # Use the model to predict using the normalized features
            numeric_prediction = model_3.predict(scaled_features)
            print(numeric_prediction)

                # Map the numerical prediction to a categorical label
            texture_prediction = stage_mapping[numeric_prediction[0]]
            #print(texture_prediction)

            hsv_image = cv2.cvtColor(image_2, cv2.COLOR_BGR2HSV)
            combined_mask = segment_spots(hsv_image)
            density = calculate_spot_density(combined_mask, hsv_image)
            # Define thresholds for density labeling
            low_threshold = 0.022  # Placeholder value for low density
            moderate_threshold = 0.17  # Placeholder value for moderate density => 0.2
            Label_2 = 'Low' if density < low_threshold else ('Moderate' if density < moderate_threshold else 'High')

            severity_mapping = {
                'Low': 'Early Stage',
                'Moderate': 'Medium Stage',
                'High': 'Severe Stage',
            }
            severity_stage_2 = severity_mapping[Label_2]
            # Check if severity stages are the same and increment the counter
            categories = ['Clustered', 'Scattered', 'Speckled', 'Varied']

                # Define severity levels
            severity_levels_spots = {
                    'Scattered': 'Early_Stage',
                    'Speckled': 'Medium_Stage',
                    'Clustered': 'Severe_Stage',
                    'Varied': 'Severe_Stage'
                }
                # Make prediction
            prediction = model_2.predict(image)

                # Convert probabilities to class label
            predicted_class_index = np.argmax(prediction)
            predicted_category = categories[predicted_class_index]

                # Map severity
            severity_Spots = severity_levels_spots.get(predicted_category, 'Unknown')

            if severity_stage_1 == severity_stage_2:
                match_counter += 1

            Total_match=WH+match_counter

            print(f'Density: {density:.4f}')
            print(f'Matching predictions count: {match_counter}')

            rgb_image = cv2.cvtColor(image_2, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(2, 2))
            plt.imshow(rgb_image)
            plt.title(f'{image_file}\n\nGreeness Level : {Label_G}-- {severity_stage_G} \n\nInfection Level : {Label_1}-- {severity_stage_1}-\n\nSpots Density: {Label_2}-- {severity_stage_2}\n\n Label_texture: {texture_prediction}\n\n Spots Distribution: {severity_Spots}')

            plt.show()

    # After processing all images, print the total count and total matches
    Total_match = WH + match_counter
    print(f'Total images processed: {count}')
    print(f'Total matching predictions: {Total_match}')
    return count, Total_match




# Specify the path to your image folder here
folder_path = f'{DATA_DIR}/All_Included_Final_Severity_Stage/Medium_Stage'
total_images, total_matches = process_images_in_folder(folder_path)





In [ ]:
# Main function to process the images in the folder
def process_images_in_folder(folder_path):
    # List all image files in the folder
    files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
    image_files = [f for f in files if f.lower().endswith(('.png','.JPG', '.jpg', '.jpeg'))]

    #Shuffle the list of image files to process them in random order
    #random.shuffle(image_files)
    match_counter=0
    WH=0
    count=0
    Total_match=0

    # Process each image file
    for image_file in image_files:
        image_path = os.path.join(folder_path, image_file)
        count = count + 1
        im = cv2.imread(image_path)
        image = load_and_resize_image(im)
        image_2 = image  # Create a copy of the original image
        # Preprocess the image
        image = np.expand_dims(image, axis=0)  # Add a batch dimension
        image = preprocess_input(image)
        # Predict the class of the image
        prediction = model.predict(image)

        # Threshold the prediction (if your model uses a sigmoid activation in the last layer)
        threshold = 0.5
        predicted_class = 'Healthy' if prediction[0][0] < threshold else 'Unhealthy'

        if predicted_class == 'Healthy':  # If the model predicts 'Healthy'
            Label_1 = 'Non-infected'
            Label_2 = 'Spotless'
            Label_G=  'No-Stress'
            severity_stage = 'NULL'
            Label_texture='NULL'
            WH +=1
            rgb_image = cv2.cvtColor(image_2, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(2, 2))
            plt.imshow(rgb_image)
            plt.title(f'{image_file}\n\nInfection Level : {Label_1}\n\nStress Level: {Label_G}\n\nSpots Density: {Label_2}\n\nSeverity:{severity_stage}\n\nTexture: {Label_texture}')

            plt.show()
        else:  # If the model predicts 'Unhealthy'
            foreground, foreground_mask = extract_foreground(image_2)
            preprocessed_foreground = preprocess_shiny(foreground)

            # Calculate the green color peak on the foreground
            green_peak = calculate_green_peak(preprocessed_foreground)
            # Set the green threshold and obtain the green mask for the foreground
            green_mask = threshold_green(foreground, green_peak)

            # Separate and refine the green and non-green regions from the foreground
            green_region, non_green_region = separate_regions(foreground, green_mask)

            # Calculate area percentages
            percentage_green, percentage_non_green, green_area, non_green_area = calculate_area_percentages(green_mask, non_green_region, foreground_mask)

            print(f'Processing image # {count} & Name :{image_file}')
            print(f"Percentage of Green Area: {percentage_green:.2f}%")
            print(f"Percentage of Non-Green Area: {percentage_non_green:.2f}%")


            # Label classification
            Label_1 = classify_plant_health(percentage_non_green)
            severity_mapping_1 = {
                'Slightly-infected': 'Early Stage',
                'Moderately-infected': 'Medium Stage',
                'Highly-infected': 'Severe Stage',
            }
            severity_stage_1 = severity_mapping_1.get(Label_1, 'Undefined Stage')

                        # Label classification
            Label_G = classify_plant_stress(percentage_green)
            severity_mapping_G = {
                'Low': 'Early Stage',
                'Medium': 'Medium Stage',
                'High': 'Severe Stage',
            }
            severity_stage_G = severity_mapping_G.get(Label_G, 'Undefined Stage')

            if non_green_region.ndim == 3:
              image3 = cv2.cvtColor(non_green_region, cv2.COLOR_BGR2GRAY)

                # Analyze texture
            results = analyze_texture(image3)
                # Define the mapping from numerical prediction to categorical label
            stage_mapping = {0:'Early_Stage', 1: 'Medium_Stage', 2: 'Severe_Stage'}

                # Convert the dictionary of features to an array-like structure
            features_array = np.array(list(results.values())).reshape(1, -1)
                # Normalize the features using the loaded scaler
            scaled_features = scaler.transform(features_array)
                #print(scaled_features.shape)  # Check the current shape
                # Assuming your model expects input of shape (1, height, width, channels)
                # You may need to reshape or adjust your input here.

                # Use the model to predict using the normalized features
                # Use the model to predict using the normalized features
            numeric_prediction = model_3.predict(scaled_features)
            print(numeric_prediction)

                # Map the numerical prediction to a categorical label
            texture_prediction = stage_mapping[numeric_prediction[0]]
            #print(texture_prediction)

            hsv_image = cv2.cvtColor(image_2, cv2.COLOR_BGR2HSV)
            combined_mask = segment_spots(hsv_image)
            density = calculate_spot_density(combined_mask, hsv_image)
            # Define thresholds for density labeling
            low_threshold = 0.022  # Placeholder value for low density
            moderate_threshold = 0.17  # Placeholder value for moderate density => 0.2
            Label_2 = 'Low' if density < low_threshold else ('Moderate' if density < moderate_threshold else 'High')

            severity_mapping = {
                'Low': 'Early Stage',
                'Moderate': 'Medium Stage',
                'High': 'Severe Stage',
            }
            severity_stage_2 = severity_mapping[Label_2]
            # Check if severity stages are the same and increment the counter
            categories = ['Clustered', 'Scattered', 'Speckled', 'Varied']

                # Define severity levels
            severity_levels_spots = {
                    'Scattered': 'Early_Stage',
                    'Speckled': 'Medium_Stage',
                    'Clustered': 'Severe_Stage',
                    'Varied': 'Severe_Stage'
                }
                # Make prediction
            prediction = model_2.predict(image)

                # Convert probabilities to class label
            predicted_class_index = np.argmax(prediction)
            predicted_category = categories[predicted_class_index]

                # Map severity
            severity_Spots = severity_levels_spots.get(predicted_category, 'Unknown')

            if severity_stage_1 == severity_stage_2:
                match_counter += 1

            Total_match=WH+match_counter

            print(f'Density: {density:.4f}')
            print(f'Matching predictions count: {match_counter}')

            rgb_image = cv2.cvtColor(image_2, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(2, 2))
            plt.imshow(rgb_image)
            plt.title(f'{image_file}\n\nGreeness Level : {Label_G}-- {severity_stage_G} \n\nInfection Level : {Label_1}-- {severity_stage_1}-\n\nSpots Density: {Label_2}-- {severity_stage_2}\n\n Label_texture: {texture_prediction}\n\n Spots Distribution: {severity_Spots}')

            plt.show()

    # After processing all images, print the total count and total matches
    Total_match = WH + match_counter
    print(f'Total images processed: {count}')
    print(f'Total matching predictions: {Total_match}')
    return count, Total_match




# Specify the path to your image folder here
folder_path = f'{DATA_DIR}/All_Included_Final_Severity_Stage/Early_Stage'
total_images, total_matches = process_images_in_folder(folder_path)





In [ ]:
# Main function to process the images in the folder
def process_images_in_folder(folder_path):
    # List all image files in the folder
    files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
    image_files = [f for f in files if f.lower().endswith(('.png','.JPG', '.jpg', '.jpeg'))]

    #Shuffle the list of image files to process them in random order
    #random.shuffle(image_files)
    match_counter=0
    WH=0
    count=0
    Total_match=0

    # Process each image file
    for image_file in image_files:
        image_path = os.path.join(folder_path, image_file)
        count = count + 1
        im = cv2.imread(image_path)
        image = load_and_resize_image(im)
        image_2 = image  # Create a copy of the original image
        # Preprocess the image
        image = np.expand_dims(image, axis=0)  # Add a batch dimension
        image = preprocess_input(image)
        # Predict the class of the image
        prediction = model.predict(image)

        # Threshold the prediction (if your model uses a sigmoid activation in the last layer)
        threshold = 0.5
        predicted_class = 'Healthy' if prediction[0][0] < threshold else 'Unhealthy'

        if predicted_class == 'Healthy':  # If the model predicts 'Healthy'
            Label_1 = 'Non-infected'
            Label_2 = 'Spotless'
            Label_G=  'No-Stress'
            severity_stage = 'NULL'
            Label_texture='NULL'
            WH +=1
            rgb_image = cv2.cvtColor(image_2, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(2, 2))
            plt.imshow(rgb_image)
            plt.title(f'{image_file}\n\nInfection Level : {Label_1}\n\nStress Level: {Label_G}\n\nSpots Density: {Label_2}\n\nSeverity:{severity_stage}\n\nTexture: {Label_texture}')

            plt.show()
        else:  # If the model predicts 'Unhealthy'
            foreground, foreground_mask = extract_foreground(image_2)
            preprocessed_foreground = preprocess_shiny(foreground)

            # Calculate the green color peak on the foreground
            green_peak = calculate_green_peak(preprocessed_foreground)
            # Set the green threshold and obtain the green mask for the foreground
            green_mask = threshold_green(foreground, green_peak)

            # Separate and refine the green and non-green regions from the foreground
            green_region, non_green_region = separate_regions(foreground, green_mask)

            # Calculate area percentages
            percentage_green, percentage_non_green, green_area, non_green_area = calculate_area_percentages(green_mask, non_green_region, foreground_mask)

            print(f'Processing image # {count} & Name :{image_file}')
            print(f"Percentage of Green Area: {percentage_green:.2f}%")
            print(f"Percentage of Non-Green Area: {percentage_non_green:.2f}%")


            # Label classification
            Label_1 = classify_plant_health(percentage_non_green)
            severity_mapping_1 = {
                'Slightly-infected': 'Early Stage',
                'Moderately-infected': 'Medium Stage',
                'Highly-infected': 'Severe Stage',
            }
            severity_stage_1 = severity_mapping_1.get(Label_1, 'Undefined Stage')

                        # Label classification
            Label_G = classify_plant_stress(percentage_green)
            severity_mapping_G = {
                'Low': 'Early Stage',
                'Medium': 'Medium Stage',
                'High': 'Severe Stage',
            }
            severity_stage_G = severity_mapping_G.get(Label_G, 'Undefined Stage')

            if non_green_region.ndim == 3:
              image3 = cv2.cvtColor(non_green_region, cv2.COLOR_BGR2GRAY)

                # Analyze texture
            results = analyze_texture(image3)
                # Define the mapping from numerical prediction to categorical label
            stage_mapping = {0:'Early_Stage', 1: 'Medium_Stage', 2: 'Severe_Stage'}

                # Convert the dictionary of features to an array-like structure
            features_array = np.array(list(results.values())).reshape(1, -1)
                # Normalize the features using the loaded scaler
            scaled_features = scaler.transform(features_array)
                #print(scaled_features.shape)  # Check the current shape
                # Assuming your model expects input of shape (1, height, width, channels)
                # You may need to reshape or adjust your input here.

                # Use the model to predict using the normalized features
                # Use the model to predict using the normalized features
            numeric_prediction = model_3.predict(scaled_features)
            print(numeric_prediction)

                # Map the numerical prediction to a categorical label
            texture_prediction = stage_mapping[numeric_prediction[0]]
            #print(texture_prediction)

            hsv_image = cv2.cvtColor(image_2, cv2.COLOR_BGR2HSV)
            combined_mask = segment_spots(hsv_image)
            density = calculate_spot_density(combined_mask, hsv_image)
            # Define thresholds for density labeling
            low_threshold = 0.022  # Placeholder value for low density
            moderate_threshold = 0.17  # Placeholder value for moderate density => 0.2
            Label_2 = 'Low' if density < low_threshold else ('Moderate' if density < moderate_threshold else 'High')

            severity_mapping = {
                'Low': 'Early Stage',
                'Moderate': 'Medium Stage',
                'High': 'Severe Stage',
            }
            severity_stage_2 = severity_mapping[Label_2]
            # Check if severity stages are the same and increment the counter
            categories = ['Clustered', 'Scattered', 'Speckled', 'Varied']

                # Define severity levels
            severity_levels_spots = {
                    'Scattered': 'Early_Stage',
                    'Speckled': 'Medium_Stage',
                    'Clustered': 'Severe_Stage',
                    'Varied': 'Severe_Stage'
                }
                # Make prediction
            prediction = model_2.predict(image)

                # Convert probabilities to class label
            predicted_class_index = np.argmax(prediction)
            predicted_category = categories[predicted_class_index]

                # Map severity
            severity_Spots = severity_levels_spots.get(predicted_category, 'Unknown')

            if severity_stage_1 == severity_stage_2:
                match_counter += 1

            Total_match=WH+match_counter

            print(f'Density: {density:.4f}')
            print(f'Matching predictions count: {match_counter}')

            rgb_image = cv2.cvtColor(image_2, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(2, 2))
            plt.imshow(rgb_image)
            plt.title(f'{image_file}\n\nGreeness Level : {Label_G}-- {severity_stage_G} \n\nInfection Level : {Label_1}-- {severity_stage_1}-\n\nSpots Density: {Label_2}-- {severity_stage_2}\n\n Label_texture: {texture_prediction}\n\n Spots Distribution: {severity_Spots}')

            plt.show()

    # After processing all images, print the total count and total matches
    Total_match = WH + match_counter
    print(f'Total images processed: {count}')
    print(f'Total matching predictions: {Total_match}')
    return count, Total_match




# Specify the path to your image folder here
folder_path = f'{DATA_DIR}/All_Included_Final_Severity_Stage/Severe_Stage'
total_images, total_matches = process_images_in_folder(folder_path)





## 16. Other exploratory dataset routines

In [ ]:
# Main function to process the images in the folder
def process_images_in_folder(root_folder_path):
    # List all subfolders in the root folder
    subfolders = [os.path.join(root_folder_path, f) for f in os.listdir(root_folder_path) if os.path.isdir(os.path.join(root_folder_path, f))]

    match_counter = 0
    WH = 0
    count = 0
    Total_match = 0

    # Continue processing until a stopping condition is met (e.g., a certain number of images have been processed)
    while True:
        # Randomly select a subfolder
        selected_folder = random.choice(subfolders)
        files = [f for f in os.listdir(selected_folder) if os.path.isfile(os.path.join(selected_folder, f))]
        image_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        if not image_files:
            continue  # Skip if the folder is empty

        # Randomly select an image file
        image_file = random.choice(image_files)
        image_path = os.path.join(selected_folder, image_file)
        count += 1

        im = cv2.imread(image_path)
        image = load_and_resize_image(im)
        image_2 = image  # Create a copy of the original image
        # Preprocess the image
        image = np.expand_dims(image, axis=0)  # Add a batch dimension
        image = preprocess_input(image)
        # Predict the class of the image
        prediction = model.predict(image)
                # Threshold the prediction (if your model uses a sigmoid activation in the last layer)
        threshold = 0.5
        predicted_class = 'Healthy' if prediction[0][0] < threshold else 'Unhealthy'

        if predicted_class == 'Healthy':  # If the model predicts 'Healthy'
            Label_1 = 'Non-infected'
            Label_2 = 'Spotless'
            Label_G=  'No-Stress'
            severity_stage = 'NULL'
            WH +=1
            rgb_image = cv2.cvtColor(image_2, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(2, 2))
            plt.imshow(rgb_image)
            plt.title(f'{image_file}\nInfection Level : {Label_1}\nStress Level: {Label_G}\nSpots Density: {Label_2}\nSeverity:{severity_stage}')

            plt.show()
        else:  # If the model predicts 'Unhealthy'
            foreground, foreground_mask = extract_foreground(image_2)
            #preprocessed_foreground = preprocess_shiny(foreground)

            # Calculate the green color peak on the foreground
            green_peak = calculate_green_peak(foreground)
            # Set the green threshold and obtain the green mask for the foreground
            green_mask = threshold_green(foreground, green_peak)

            # Separate and refine the green and non-green regions from the foreground
            green_region, non_green_region = separate_regions(foreground, green_mask)

            # Calculate area percentages
            percentage_green, percentage_non_green, green_area, non_green_area = calculate_area_percentages(green_mask, non_green_region, foreground_mask)

            print(f'Processing image # {count} & Name :{image_file}')
            print(f"Percentage of Green Area: {percentage_green:.2f}%")
            print(f"Percentage of Non-Green Area: {percentage_non_green:.2f}%")


            # Label classification
            Label_1 = classify_plant_health(percentage_non_green)
            severity_mapping_1 = {
                'Slightly-infected': 'Early Stage',
                'Moderately-infected': 'Medium Stage',
                'Highly-infected': 'Severe Stage',
            }
            severity_stage_1 = severity_mapping_1.get(Label_1, 'Undefined Stage')

                        # Label classification
            Label_G = classify_plant_stress(percentage_green)
            severity_mapping_G = {
                'Low': 'Early Stage',
                'Medium': 'Medium Stage',
                'High': 'Severe Stage',
            }
            severity_stage_G = severity_mapping_G.get(Label_G, 'Undefined Stage')

            hsv_image = cv2.cvtColor(image_2, cv2.COLOR_BGR2HSV)
            combined_mask = segment_spots(hsv_image)
            density = calculate_spot_density(combined_mask, hsv_image)

            # Define thresholds for density labeling
            low_threshold = 0.02  # Placeholder value for low density
            moderate_threshold = 0.1  # Placeholder value for moderate density
            Label_2 = 'Low' if density < low_threshold else ('Moderate' if density < moderate_threshold else 'High')

            severity_mapping = {
                'Low': 'Early Stage',
                'Moderate': 'Medium Stage',
                'High': 'Severe Stage',
            }
            severity_stage_2 = severity_mapping[Label_2]
            # Check if severity stages are the same and increment the counter
                            #Check Spots Distribution
            categories = ['Clustered', 'Scattered', 'Speckled', 'Varied']

                # Define severity levels
            severity_levels_spots = {
                    'Scattered': 'Early stage',
                    'Speckled': 'Medium stage',
                    'Clustered': 'Severe stage',
                    'Varied': 'Severe stage'
                }
                # Make prediction
            prediction = model_2.predict(image)

                # Convert probabilities to class label
            predicted_class_index = np.argmax(prediction)
            predicted_category = categories[predicted_class_index]

                # Map severity
            severity_Spots = severity_levels_spots.get(predicted_category, 'Unknown')
            if severity_stage_1 == severity_stage_2:
              match_counter += 1

            Total_match=WH+match_counter








            print(f'Density: {density:.4f}')
            #print(f'Matching predictions count: {match_counter}')

            rgb_image = cv2.cvtColor(image_2, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(2, 2))
            plt.imshow(rgb_image)
            plt.title(f'{image_file}\nGreeness Level : {Label_G}-- {severity_stage_G}\nInfection Level : {Label_1}-- {severity_stage_1}-\nSpots Density: {Label_2}-- {severity_stage_2}\n--\nSpots Distribution: {predicted_category}-- {severity_Spots}\n')

            plt.show()



        # Include some stopping condition, for example, a certain number of images
        if count >= 6000:  # Replace 'some_number' with the desired number of images to process
            break

    # After processing all images, print the total count and total matches

    print(f'Total Healthy images processed: {WH}')
    print(f'Total Severity matching predictions for all 3 cases: {match_counter}')
    print(f'Total  Matching predictions for all image: {Total_match}')
    return count, Total_match

# Specify the path to your image folder here
root_folder_path = f"{DATA_DIR}/Severity_Stage_Spots/Severity_Stage_Spots"
total_images, total_matches = process_images_in_folder(root_folder_path)


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random
from sklearn.metrics import accuracy_score

# Assumed predefined functions:
# extract_foreground, preprocess_shiny, calculate_green_peak, threshold_green, separate_regions,
# calculate_area_percentages, classify_plant_health, classify_plant_stress, segment_spots, calculate_spot_density,
# load_and_resize_image, preprocess_input, model

def process_images_in_folder(root_folder_path):
    subfolders = [os.path.join(root_folder_path, f) for f in os.listdir(root_folder_path) if os.path.isdir(os.path.join(root_folder_path, f))]
    health_labels = []
    health_predictions = []
    stress_labels = []
    stress_predictions = []
    density_labels = []
    density_predictions = []
    spots_labels = []
    spots_predictions = []
    final_predictions = []  # Store final predictions after majority voting
    actual_labels = []  # Store actual labels for accuracy calculation
    count = 0
    # Variables for binary classification (Healthy vs. Unhealthy)
    binary_predictions = []
    binary_actual_labels = []

    for subfolder in subfolders:
        actual_stage = os.path.basename(subfolder)  # The actual stage is the subfolder name

        files = [f for f in os.listdir(subfolder) if os.path.isfile(os.path.join(subfolder, f))]
        image_files = [f for f in files if f.lower().endswith(('.png', '.jpg','.JPG', '.jpeg'))]

        for image_file in image_files:
            image_path = os.path.join(subfolder, image_file)
            count += 1

            im = cv2.imread(image_path)
            image = load_and_resize_image(im)
            image_2 = image.copy()
            image = np.expand_dims(image, axis=0)
            image = preprocess_input(image)
            prediction = model.predict(image)
            threshold = 0.5
            predicted_class = 'Healthy' if prediction[0][0] < threshold else 'Unhealthy'
            # Record binary predictions and actual labels

            binary_actual_labels.append(actual_binary_label)
            binary_predictions.append(predicted_class)

            if predicted_class == 'Healthy':
                Label_1 = 'Non-infected'
                Label_2 = 'Spotless'
                Label_G = 'No-Stress'
                severity_levels_spots='No-Spots'
                # Assuming that 'NULL' maps to 'Healthy'
                health_predictions.append('NULL')  # Or 'Healthy' if you have such a label
                stress_predictions.append('NULL')  # Same assumption
                density_predictions.append('NULL')  # Same assumption
                spots_predictions.append('NULL')  # Same assumption
            else:
                foreground, foreground_mask = extract_foreground(image_2)
                preprocessed_foreground = preprocess_shiny(foreground)

                # Calculate the green color peak on the foreground
                green_peak = calculate_green_peak(preprocessed_foreground)
                # Set the green threshold and obtain the green mask for the foreground
                green_mask = threshold_green(foreground, green_peak)

                # Separate and refine the green and non-green regions from the foreground
                green_region, non_green_region = separate_regions(foreground, green_mask)

                # Calculate area percentages
                percentage_green, percentage_non_green, green_area, non_green_area = calculate_area_percentages(green_mask, non_green_region, foreground_mask)
                 # Label classification
                Label_1 = classify_plant_health(percentage_non_green)
                severity_mapping_1 = {
                    'Slightly-infected': 'Early Stage',
                    'Moderately-infected': 'Medium Stage',
                    'Highly-infected': 'Severe Stage',
                }
                severity_stage_1 = severity_mapping_1.get(Label_1, 'Undefined Stage')

                            # Label classification
                Label_G = classify_plant_stress(percentage_green)
                severity_mapping_G = {
                    'Low': 'Early Stage',
                    'Medium': 'Medium Stage',
                    'High': 'Severe Stage',
                }
                severity_stage_G = severity_mapping_G.get(Label_G, 'Undefined Stage')

                hsv_image = cv2.cvtColor(image_2, cv2.COLOR_BGR2HSV)
                combined_mask = segment_spots(hsv_image)
                density = calculate_spot_density(combined_mask, hsv_image)

                # Define thresholds for density labeling
                low_threshold = 0.023  # Placeholder value for low density->0.022
                moderate_threshold = 0.18  # Placeholder value for moderate density ->0.17
                Label_2 = 'Low' if density < low_threshold else ('Moderate' if density < moderate_threshold else 'High')

                severity_mapping = {
                    'Low': 'Early Stage',
                    'Moderate': 'Medium Stage',
                    'High': 'Severe Stage',
                }

                severity_stage_2 = severity_mapping[Label_2]
                # Define categories
                #Check Spots Distribution
                categories = ['Clustered', 'Scattered', 'Speckled', 'Varied']

                # Define severity levels
                severity_levels_spots = {
                    'Scattered': 'Early stage',
                    'Speckled': 'Medium stage',
                    'Clustered': 'Severe stage',
                    'Varied': 'Severe stage'
                }
                # Make prediction
                prediction = model_2.predict(image_2)

                # Convert probabilities to class label
                predicted_class_index = np.argmax(prediction)
                predicted_category = categories[predicted_class_index]

                # Map severity
                severity_Spots = severity_levels_spots.get(predicted_category, 'Unknown')


            # Check if severity stages are the same and increment the counter
                health_predictions.append(severity_stage_1)
                stress_predictions.append(severity_stage_G)
                density_predictions.append(severity_stage_2)
                spots_predictions.append(severity_Spots)

            # Add actual labels for accuracy calculation
            health_labels.append(actual_stage)
            stress_labels.append(actual_stage)
            density_labels.append(actual_stage)
            spots_labels.append(actual_stage)

            # Add actual label for accuracy calculation
            actual_labels.append(actual_stage)

            # Majority voting to determine final prediction
            final_prediction = majority_voting(severity_stage_1, severity_stage_G, severity_stage_2,severity_Spots)
            final_predictions.append(final_prediction)



            if count >= 6000:  # Stopping condition
                break

        if count >= 6000:
            break


    # Accuracy calculations
    health_accuracy = accuracy_score(health_labels, health_predictions)
    print('\n\n Infection Level Confusion Matrix')
    class_labels = ['Early Stage', 'Medium Stage', 'Severe Stage']  # Adjust labels as necessary
    plot_confusion_matrix(health_labels, health_predictions, class_labels)

# Then follow with your accuracy calculations and printing
    print('\n\n Leaf Stress Confusion Matrix')
    stress_accuracy = accuracy_score(stress_labels, stress_predictions)
    plot_confusion_matrix(stress_labels, stress_predictions, class_labels)

    print('\n\n Spots Density Confusion Matrix')
    density_accuracy = accuracy_score(density_labels, density_predictions)
    plot_confusion_matrix(density_labels, density_predictions, class_labels)

    print('\n\n Spots Distribution Confusion Matrix')
    spots_accuracy = accuracy_score(spots_labels, spots_predictions)
    plot_confusion_matrix(spots_labels, spots_predictions, class_labels)

    print('\n\n Majority Voting Confusion Matrix')
    # Calculate and print accuracy
    final_accuracy = accuracy_score(actual_labels, final_predictions)
    plot_confusion_matrix(actual_labels, final_predictions, class_labels)

    # Visualization call (add this)


    print(f'Infection Level Accuracy: {health_accuracy}')
    print(f'Stress Classification Accuracy: {stress_accuracy}')
    print(f'Density Classification Accuracy: {density_accuracy}')
    print(f'Spots Distribution Accuracy: {spots_accuracy}')
    print(f'Final Classification Accuracy: {final_accuracy}')
    print(f'Total Image: {count}')

    return {
        'total_images_processed': count,
        'health_accuracy': health_accuracy,
        'stress_accuracy': stress_accuracy,
        'density_accuracy': density_accuracy,
        'spots_accuracy': spots_accuracy,
        'final_accuracy': final_accuracy
    }
# Specify the path to your image folder here
root_folder_path = f'{DATA_DIR}/Severity_Stage_Labeling'
results = process_images_in_folder(root_folder_path)

In [ ]:
# Main function to process the images in the folder
def process_images_in_folder(folder_path):
    # List all image files in the folder
    files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
    image_files = [f for f in files if f.lower().endswith(('.png','.JPG', '.jpg', '.jpeg'))]

    #Shuffle the list of image files to process them in random order
    #random.shuffle(image_files)
    match_counter=0
    WH=0
    count=0
    Total_match=0

    # Process each image file
    for image_file in image_files:
        image_path = os.path.join(folder_path, image_file)
        count = count + 1
        im = cv2.imread(image_path)
        image = load_and_resize_image(im)
        image_2 = image  # Create a copy of the original image
        # Preprocess the image
        image = np.expand_dims(image, axis=0)  # Add a batch dimension
        image = preprocess_input(image)
        # Predict the class of the image
        prediction = model.predict(image)

        # Threshold the prediction (if your model uses a sigmoid activation in the last layer)
        threshold = 0.5
        predicted_class = 'Healthy' if prediction[0][0] < threshold else 'Unhealthy'

        if predicted_class == 'Healthy':  # If the model predicts 'Healthy'
            Label_1 = 'Non-infected'
            Label_2 = 'Spotless'
            Label_G=  'No-Stress'
            severity_stage = 'NULL'
            WH +=1
            rgb_image = cv2.cvtColor(image_2, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(2, 2))
            plt.imshow(rgb_image)
            plt.title(f'{image_file}\n\nInfection Level : {Label_1}\n\nStress Level: {Label_G}\n\nSpots Density: {Label_2}\n\nSeverity:{severity_stage}')

            plt.show()
        else:  # If the model predicts 'Unhealthy'
            foreground, foreground_mask = extract_foreground(image_2)
            preprocessed_foreground = preprocess_shiny(foreground)

            # Calculate the green color peak on the foreground
            green_peak = calculate_green_peak(preprocessed_foreground)
            # Set the green threshold and obtain the green mask for the foreground
            green_mask = threshold_green(foreground, green_peak)

            # Separate and refine the green and non-green regions from the foreground
            green_region, non_green_region = separate_regions(foreground, green_mask)

            # Calculate area percentages
            percentage_green, percentage_non_green, green_area, non_green_area = calculate_area_percentages(green_mask, non_green_region, foreground_mask)

            print(f'Processing image # {count} & Name :{image_file}')
            print(f"Percentage of Green Area: {percentage_green:.2f}%")
            print(f"Percentage of Non-Green Area: {percentage_non_green:.2f}%")


            # Label classification
            Label_1 = classify_plant_health(percentage_non_green)
            severity_mapping_1 = {
                'Slightly-infected': 'Early Stage',
                'Moderately-infected': 'Medium Stage',
                'Highly-infected': 'Severe Stage',
            }
            severity_stage_1 = severity_mapping_1.get(Label_1, 'Undefined Stage')

                        # Label classification
            Label_G = classify_plant_stress(percentage_green)
            severity_mapping_G = {
                'Low': 'Early Stage',
                'Medium': 'Medium Stage',
                'High': 'Severe Stage',
            }
            severity_stage_G = severity_mapping_G.get(Label_G, 'Undefined Stage')

            hsv_image = cv2.cvtColor(image_2, cv2.COLOR_BGR2HSV)
            combined_mask = segment_spots(hsv_image)
            density = calculate_spot_density(combined_mask, hsv_image)
            # Define thresholds for density labeling
            low_threshold = 0.022  # Placeholder value for low density
            moderate_threshold = 0.17  # Placeholder value for moderate density => 0.2
            Label_2 = 'Low' if density < low_threshold else ('Moderate' if density < moderate_threshold else 'High')

            severity_mapping = {
                'Low': 'Early Stage',
                'Moderate': 'Medium Stage',
                'High': 'Severe Stage',
            }
            severity_stage_2 = severity_mapping[Label_2]
            # Check if severity stages are the same and increment the counter
            if severity_stage_1 == severity_stage_2:
                match_counter += 1

            Total_match=WH+match_counter

            print(f'Density: {density:.4f}')
            print(f'Matching predictions count: {match_counter}')

            rgb_image = cv2.cvtColor(image_2, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(2, 2))
            plt.imshow(rgb_image)
            plt.title(f'{image_file}\n\nGreeness Level : {Label_G}-- {severity_stage_G} \n\nInfection Level : {Label_1}-- {severity_stage_1}-\n\nSpots Density: {Label_2}-- {severity_stage_2}\n')

            plt.show()

    # After processing all images, print the total count and total matches
    Total_match = WH + match_counter
    print(f'Total images processed: {count}')
    print(f'Total matching predictions: {Total_match}')
    return count, Total_match




# Specify the path to your image folder here
folder_path = f'{DATA_DIR}/Severity_Stage_Val/Medium_Stage'
total_images, total_matches = process_images_in_folder(folder_path)





## 17. Single-image routine

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Assuming the necessary model loading and preprocessing functions, as well as image analysis functions,
# are defined elsewhere in your codebase.

def process_single_image(image_path):
    if not os.path.isfile(image_path) or not image_path.lower().endswith(('.png','.JPG', '.jpg', '.jpeg')):
        print("The provided file is not an image or does not exist.")
        return

    count = 1
    WH = 0
    match_counter = 0

    im = cv2.imread(image_path)
    image = load_and_resize_image(im)  # This function needs to be defined as per your model's requirements
    image_2 = image.copy()  # Create a copy of the original image for further processing
    # Preprocess the image
    image = np.expand_dims(image, axis=0)  # Add a batch dimension
    image = preprocess_input(image)  # Adjust according to your model's preprocessing needs
    # Predict the class of the image
    prediction = model.predict(image)
    threshold = 0.5
    predicted_class = 'Healthy' if prediction[0][0] < threshold else 'Unhealthy'

    if predicted_class == 'Healthy':
        Label_1 = 'Non-infected'
        Label_2 = 'Spotless'
        Label_G = 'No-Stress'
        severity_stage = 'NULL'
        WH += 1
        display_image(image_2, image_path, Label_1, Label_2, Label_G, severity_stage)
    else:
        # If the model predicts 'Unhealthy', process accordingly
        process_unhealthy_image(image_2, image_path, count)

    Total_match = WH + match_counter
    print(f'Total images processed: {count}')
    print(f'Total matching predictions: {Total_match}')
    return count, Total_match


def process_unhealthy_image(image, image_path, count):
    print(f'Processing image # {count} & Name: {os.path.basename(image_path)}')

    # Extract the foreground from the image (assuming you have a function for this)
    foreground, foreground_mask = extract_foreground(image)

    # Calculate the green color peak in the foreground (define this function based on your analysis)
    green_peak = calculate_green_peak(foreground)

    # Threshold the image based on the green color peak to get a green mask (define this function)
    green_mask = threshold_green(foreground, green_peak)

    # Separate and refine green and non-green regions (this function should be defined by you)
    green_region, non_green_region = separate_regions(foreground, green_mask)

    # Calculate the percentage of green and non-green areas (another function you need to define)
    percentage_green, percentage_non_green, green_area, non_green_area = calculate_area_percentages(green_mask, non_green_region, foreground_mask)

    # Classify the plant's health based on the non-green percentage
    Label_1 = classify_plant_health(percentage_non_green)

    # Determine the severity stage based on your classifications
    severity_mapping_1 = {
        'Slightly-infected': 'Early Stage',
        'Moderately-infected': 'Medium Stage',
        'Highly-infected': 'Severe Stage',
    }
    severity_stage_1 = severity_mapping_1.get(Label_1, 'Undefined Stage')

    # Classify the plant's stress level based on the green percentage
    Label_G = classify_plant_stress(percentage_green)

    # Determine the severity stage for stress
    severity_mapping_G = {
        'Low': 'Early Stage',
        'Medium': 'Medium Stage',
        'High': 'Severe Stage',
    }
    severity_stage_G = severity_mapping_G.get(Label_G, 'Undefined Stage')

    # Segment spots if applicable and calculate spot density
    hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    combined_mask = segment_spots(hsv_image)
    density = calculate_spot_density(combined_mask, hsv_image)

    # Define thresholds for density labeling
    low_threshold = 0.022  # Example threshold, adjust as needed
    moderate_threshold = 0.17  # Example threshold, adjust as needed
    Label_2 = 'Low' if density < low_threshold else ('Moderate' if density < moderate_threshold else 'High')

    severity_mapping_2 = {
        'Low': 'Early Stage',
        'Moderate': 'Medium Stage',
        'High': 'Severe Stage',
    }
    severity_stage_2 = severity_mapping_2[Label_2]

    # Display the image with the calculated labels and severity stages
    display_image(image, image_path, Label_1, Label_2, Label_G, severity_stage_1, severity_stage_G, severity_stage_2)

def display_image(image, image_path, Label_1, Label_2, Label_G, severity_stage_1, severity_stage_G, severity_stage_2):
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(4, 4))
    plt.imshow(rgb_image)
    plt.title(f'{os.path.basename(image_path)}\nInfection Level: {Label_1} -- {severity_stage_1}\nStress Level: {Label_G} -- {severity_stage_G}\nSpots Density: {Label_2} -- {severity_stage_2}')
    plt.show()


# Example usage
image_path = f'{DATA_DIR}/StrawberryLeafScorch(3092).JPG'  # Replace with the actual path to your image
total_images, total_matches = process_single_image(image_path)
